In [12]:
from datetime import datetime, timedelta

fecha_fin = "2026-08-07"
hoy = datetime.now()
fecha_inicio_valida = hoy - timedelta(days=729)

print(f"Hoy es: {hoy.strftime('%Y-%m-%d')}")
print(f"La fecha de inicio más antigua permitida es: {fecha_inicio_valida.strftime('%Y-%m-%d')}")

Hoy es: 2026-08-14
La fecha de inicio más antigua permitida es: 2024-08-15


In [13]:
import yfinance as yf

datos_h1 = yf.download("GC=F", start=fecha_inicio_valida.strftime('%Y-%m-%d'), end=fecha_fin, interval="1h")

datos_h1.columns = datos_h1.columns.get_level_values(0)
datos_h1.columns.name = None
datos_h1 = datos_h1[["Open", "High", "Low", "Close", "Volume"]]

[*********************100%***********************]  1 of 1 completed


In [14]:
import pandas as pd

datos_h1_desplazado = datos_h1.copy()
datos_h1_desplazado.index = datos_h1_desplazado.index - pd.Timedelta(hours=17)

datos_w = datos_h1_desplazado.resample("W-FRI").agg({
    "Open": "first", "High": "max", "Low": "min", "Close": "last", "Volume": "sum"
}).dropna()

datos_w.index = datos_w.index + pd.Timedelta(hours=17)

datos_w.head()

,Open,High,Low,Close,Volume
Datetime,,,,,
2024-08-16 17:00:00-04:00,2491.300049,2548.300049,2469.199951,2546.199951,355372
2024-08-23 17:00:00-04:00,2549.699951,2570.399902,2506.399902,2548.699951,805134
2024-08-30 17:00:00-04:00,2545.100098,2564.300049,2526.600098,2535.899902,742815
2024-09-06 17:00:00-04:00,2536.000000,2559.800049,2502.699951,2526.800049,719395
2024-09-13 17:00:00-04:00,2526.500000,2614.399902,2514.199951,2606.199951,856980


In [15]:
datos_w.to_csv("../data/XAUUSD_w_2024_2026.csv")
print("Datos semanales guardados correctamente")

Datos semanales guardados correctamente


In [16]:
datos_w["C1"] = datos_w["Close"]
datos_w["C2"] = datos_w["Close"].shift(1)
datos_w["C3"] = datos_w["Close"].shift(2)

datos_w["Escenario_A"] = (datos_w["C1"] > datos_w["C2"]) & (datos_w["C1"] > datos_w["C3"])
datos_w["Escenario_B"] = (datos_w["C1"] < datos_w["C2"]) & (datos_w["C1"] > datos_w["C3"])

datos_w["Es_roja"] = datos_w["Close"] < datos_w["Open"]
datos_w["OURAS"] = datos_w["Open"].where(datos_w["Es_roja"]).ffill()

datos_w["Sobre_OURAS_C1"] = datos_w["C1"] > datos_w["OURAS"]
datos_w["Sobre_OURAS_C2"] = datos_w["C2"] > datos_w["OURAS"]
datos_w["Sobre_OURAS_C3"] = datos_w["C3"] > datos_w["OURAS"]

datos_w["Conteo_sobre_OURAS"] = datos_w["Sobre_OURAS_C1"].astype(int) + datos_w["Sobre_OURAS_C2"].astype(int) + datos_w["Sobre_OURAS_C3"].astype(int)

datos_w["Sub_caso"] = "N/A"
datos_w.loc[datos_w["Conteo_sobre_OURAS"] == 3, "Sub_caso"] = "1_Limpio"
datos_w.loc[datos_w["Conteo_sobre_OURAS"] == 0, "Sub_caso"] = "2_Sin_cruzar"
datos_w.loc[(datos_w["Conteo_sobre_OURAS"] == 1) | (datos_w["Conteo_sobre_OURAS"] == 2), "Sub_caso"] = "3_Cruzando"

datos_w["Combinacion"] = "Ninguno"
datos_w.loc[datos_w["Escenario_A"], "Combinacion"] = "A_" + datos_w["Sub_caso"]
datos_w.loc[datos_w["Escenario_B"], "Combinacion"] = "B_" + datos_w["Sub_caso"]

datos_w[["Close", "C1", "C2", "C3", "OURAS", "Combinacion"]].tail(15)


,Close,C1,C2,C3,OURAS,Combinacion
Datetime,,,,,,
2026-05-01 17:00:00-04:00,4644.500000,4644.500000,4740.899902,4879.600098,4696.100098,Ninguno
2026-05-08 17:00:00-04:00,4730.700195,4730.700195,4644.500000,4740.899902,4696.100098,Ninguno
2026-05-15 17:00:00-04:00,4561.899902,4561.899902,4730.700195,4644.500000,4696.799805,Ninguno
2026-05-22 17:00:00-04:00,4523.200195,4523.200195,4561.899902,4730.700195,4557.399902,Ninguno
2026-05-29 17:00:00-04:00,4593.000000,4593.000000,4523.200195,4561.899902,4557.399902,A_3_Cruzando
2026-06-05 17:00:00-04:00,4365.299805,4365.299805,4593.000000,4523.200195,4555.299805,Ninguno
2026-06-12 17:00:00-04:00,4238.799805,4238.799805,4365.299805,4593.000000,4342.200195,Ninguno
2026-06-19 17:00:00-04:00,4167.500000,4167.500000,4238.799805,4365.299805,4325.600098,Ninguno
2026-06-26 17:00:00-04:00,4096.299805,4096.299805,4167.500000,4238.799805,4163.899902,Ninguno


In [17]:
datos_w["Escenario_A_bajista"] = (datos_w["C1"] < datos_w["C2"]) & (datos_w["C1"] < datos_w["C3"])
datos_w["Escenario_B_bajista"] = (datos_w["C1"] > datos_w["C2"]) & (datos_w["C1"] < datos_w["C3"])

datos_w["Es_verde"] = datos_w["Close"] > datos_w["Open"]
datos_w["OUVAS"] = datos_w["Open"].where(datos_w["Es_verde"]).ffill()

datos_w["Debajo_OUVAS_C1"] = datos_w["C1"] < datos_w["OUVAS"]
datos_w["Debajo_OUVAS_C2"] = datos_w["C2"] < datos_w["OUVAS"]
datos_w["Debajo_OUVAS_C3"] = datos_w["C3"] < datos_w["OUVAS"]

datos_w["Conteo_debajo_OUVAS"] = datos_w["Debajo_OUVAS_C1"].astype(int) + datos_w["Debajo_OUVAS_C2"].astype(int) + datos_w["Debajo_OUVAS_C3"].astype(int)

datos_w["Sub_caso_bajista"] = "N/A"
datos_w.loc[datos_w["Conteo_debajo_OUVAS"] == 3, "Sub_caso_bajista"] = "1_Limpio"
datos_w.loc[datos_w["Conteo_debajo_OUVAS"] == 0, "Sub_caso_bajista"] = "2_Sin_cruzar"
datos_w.loc[(datos_w["Conteo_debajo_OUVAS"] == 1) | (datos_w["Conteo_debajo_OUVAS"] == 2), "Sub_caso_bajista"] = "3_Cruzando"

datos_w["Combinacion_bajista"] = "Ninguno"
datos_w.loc[datos_w["Escenario_A_bajista"], "Combinacion_bajista"] = "A_" + datos_w["Sub_caso_bajista"]
datos_w.loc[datos_w["Escenario_B_bajista"], "Combinacion_bajista"] = "B_" + datos_w["Sub_caso_bajista"]

datos_w[["Close", "C1", "C2", "C3", "OUVAS", "Combinacion_bajista"]].tail(15)

,Close,C1,C2,C3,OUVAS,Combinacion_bajista
Datetime,,,,,,
2026-05-01 17:00:00-04:00,4644.500000,4644.500000,4740.899902,4879.600098,4688.700195,A_3_Cruzando
2026-05-08 17:00:00-04:00,4730.700195,4730.700195,4644.500000,4740.899902,4633.299805,B_2_Sin_cruzar
2026-05-15 17:00:00-04:00,4561.899902,4561.899902,4730.700195,4644.500000,4633.299805,A_3_Cruzando
2026-05-22 17:00:00-04:00,4523.200195,4523.200195,4561.899902,4730.700195,4633.299805,A_3_Cruzando
2026-05-29 17:00:00-04:00,4593.000000,4593.000000,4523.200195,4561.899902,4542.700195,Ninguno
2026-06-05 17:00:00-04:00,4365.299805,4365.299805,4593.000000,4523.200195,4542.700195,A_3_Cruzando
2026-06-12 17:00:00-04:00,4238.799805,4238.799805,4365.299805,4593.000000,4542.700195,A_3_Cruzando
2026-06-19 17:00:00-04:00,4167.500000,4167.500000,4238.799805,4365.299805,4542.700195,A_1_Limpio
2026-06-26 17:00:00-04:00,4096.299805,4096.299805,4167.500000,4238.799805,4542.700195,A_1_Limpio


In [18]:
conteo_alcista = datos_w[datos_w["Combinacion"] != "Ninguno"]["Combinacion"].value_counts()
conteo_bajista = datos_w[datos_w["Combinacion_bajista"] != "Ninguno"]["Combinacion_bajista"].value_counts()

print("--- Escenarios ALCISTAS (semanal) ---")
print(conteo_alcista)
print(f"Total alcistas: {conteo_alcista.sum()}")

print("\n--- Escenarios BAJISTAS (semanal) ---")
print(conteo_bajista)
print(f"Total bajistas: {conteo_bajista.sum()}")

print(f"\nTotal general (alcistas + bajistas): {conteo_alcista.sum() + conteo_bajista.sum()}")
print(f"Total de velas semanales disponibles: {len(datos_w)}")

--- Escenarios ALCISTAS (semanal) ---
Combinacion
A_3_Cruzando      26
A_1_Limpio        18
B_2_Sin_cruzar     7
B_3_Cruzando       6
A_2_Sin_cruzar     5
B_1_Limpio         1
Name: count, dtype: int64
Total alcistas: 63

--- Escenarios BAJISTAS (semanal) ---
Combinacion_bajista
A_3_Cruzando      19
A_2_Sin_cruzar     6
B_2_Sin_cruzar     6
B_3_Cruzando       6
A_1_Limpio         2
Name: count, dtype: int64
Total bajistas: 39

Total general (alcistas + bajistas): 102
Total de velas semanales disponibles: 104


In [19]:
datos_w["Fecha_apertura"] = datos_w.index - pd.Timedelta(days=5)
datos_w["Fecha_cierre"] = datos_w.index

datos_w[["Fecha_apertura", "Fecha_cierre", "Combinacion", "Combinacion_bajista"]].tail(15)

,Fecha_apertura,Fecha_cierre,Combinacion,Combinacion_bajista
Datetime,,,,
2026-05-01 17:00:00-04:00,2026-04-26 17:00:00-04:00,2026-05-01 17:00:00-04:00,Ninguno,A_3_Cruzando
2026-05-08 17:00:00-04:00,2026-05-03 17:00:00-04:00,2026-05-08 17:00:00-04:00,Ninguno,B_2_Sin_cruzar
2026-05-15 17:00:00-04:00,2026-05-10 17:00:00-04:00,2026-05-15 17:00:00-04:00,Ninguno,A_3_Cruzando
2026-05-22 17:00:00-04:00,2026-05-17 17:00:00-04:00,2026-05-22 17:00:00-04:00,Ninguno,A_3_Cruzando
2026-05-29 17:00:00-04:00,2026-05-24 17:00:00-04:00,2026-05-29 17:00:00-04:00,A_3_Cruzando,Ninguno
2026-06-05 17:00:00-04:00,2026-05-31 17:00:00-04:00,2026-06-05 17:00:00-04:00,Ninguno,A_3_Cruzando
2026-06-12 17:00:00-04:00,2026-06-07 17:00:00-04:00,2026-06-12 17:00:00-04:00,Ninguno,A_3_Cruzando
2026-06-19 17:00:00-04:00,2026-06-14 17:00:00-04:00,2026-06-19 17:00:00-04:00,Ninguno,A_1_Limpio
2026-06-26 17:00:00-04:00,2026-06-21 17:00:00-04:00,2026-06-26 17:00:00-04:00,Ninguno,A_1_Limpio


In [20]:
datos_h1_desplazado = datos_h1.copy()
datos_h1_desplazado.index = datos_h1_desplazado.index - pd.Timedelta(hours=17)

datos_h12 = datos_h1_desplazado.resample("12h").agg({
    "Open": "first", "High": "max", "Low": "min", "Close": "last", "Volume": "sum"
}).dropna()

datos_h12.index = datos_h12.index + pd.Timedelta(hours=17)

datos_h12["C1"] = datos_h12["Close"]
datos_h12["C2"] = datos_h12["Close"].shift(1)
datos_h12["C3"] = datos_h12["Close"].shift(2)

datos_h12["Escenario_A"] = (datos_h12["C1"] > datos_h12["C2"]) & (datos_h12["C1"] > datos_h12["C3"])
datos_h12["Escenario_B"] = (datos_h12["C1"] < datos_h12["C2"]) & (datos_h12["C1"] > datos_h12["C3"])

datos_h12["Es_roja"] = datos_h12["Close"] < datos_h12["Open"]
datos_h12["OURAS"] = datos_h12["Open"].where(datos_h12["Es_roja"]).ffill()

datos_h12["Sobre_OURAS_C1"] = datos_h12["C1"] > datos_h12["OURAS"]
datos_h12["Sobre_OURAS_C2"] = datos_h12["C2"] > datos_h12["OURAS"]
datos_h12["Sobre_OURAS_C3"] = datos_h12["C3"] > datos_h12["OURAS"]
datos_h12["Conteo_sobre_OURAS"] = datos_h12["Sobre_OURAS_C1"].astype(int) + datos_h12["Sobre_OURAS_C2"].astype(int) + datos_h12["Sobre_OURAS_C3"].astype(int)

datos_h12["Sub_caso"] = "N/A"
datos_h12.loc[datos_h12["Conteo_sobre_OURAS"] == 3, "Sub_caso"] = "1_Limpio"
datos_h12.loc[datos_h12["Conteo_sobre_OURAS"] == 0, "Sub_caso"] = "2_Sin_cruzar"
datos_h12.loc[(datos_h12["Conteo_sobre_OURAS"] == 1) | (datos_h12["Conteo_sobre_OURAS"] == 2), "Sub_caso"] = "3_Cruzando"

datos_h12["Combinacion"] = "Ninguno"
datos_h12.loc[datos_h12["Escenario_A"], "Combinacion"] = "A_" + datos_h12["Sub_caso"]
datos_h12.loc[datos_h12["Escenario_B"], "Combinacion"] = "B_" + datos_h12["Sub_caso"]

datos_h12["Retorno_vela_siguiente"] = datos_h12["Close"].pct_change().shift(-1)
datos_h12["Siguiente_fue_alcista"] = datos_h12["Close"].shift(-1) > datos_h12["Open"].shift(-1)

In [21]:
datos_h12_sorted = datos_h12.sort_index()
datos_w_sorted = datos_w.sort_index()

datos_h12_sorted.index = datos_h12_sorted.index.as_unit("us")
datos_w_sorted.index = datos_w_sorted.index.as_unit("us")

comparacion = pd.merge_asof(
    datos_h12_sorted,
    datos_w_sorted[["Combinacion"]].rename(columns={"Combinacion": "Combinacion_semanal"}),
    left_index=True, right_index=True,
    direction="backward"
)


In [22]:
import pandas as pd
from scipy import stats

datos_h12_sorted = datos_h12.sort_index()
datos_w_sorted = datos_w.sort_index()

datos_h12_sorted.index = datos_h12_sorted.index.astype("datetime64[us, America/New_York]")
datos_w_sorted.index = datos_w_sorted.index.astype("datetime64[us, America/New_York]")

comparacion = pd.merge_asof(
    datos_h12_sorted,
    datos_w_sorted[["Combinacion"]].rename(columns={"Combinacion": "Combinacion_semanal"}),
    left_index=True, right_index=True,
    direction="backward"
)

comparacion.head()

,Open,High,Low,Close,Volume,C1,C2,C3,Escenario_A,Escenario_B,...,OURAS,Sobre_OURAS_C1,Sobre_OURAS_C2,Sobre_OURAS_C3,Conteo_sobre_OURAS,Sub_caso,Combinacion,Retorno_vela_siguiente,Siguiente_fue_alcista,Combinacion_semanal
Datetime,,,,,,,,,,,,,,,,,,,,,
2024-08-14 17:00:00-04:00,2491.300049,2496.399902,2487.699951,2493.000000,17633,2493.000000,NaN,NaN,False,False,...,NaN,False,False,False,0,2_Sin_cruzar,Ninguno,0.000602,True,NaN
2024-08-15 05:00:00-04:00,2493.000000,2508.000000,2469.199951,2494.500000,124534,2494.500000,2493.000000,NaN,False,False,...,NaN,False,False,False,0,2_Sin_cruzar,Ninguno,0.002205,True,NaN
2024-08-15 17:00:00-04:00,2494.399902,2501.899902,2488.199951,2500.000000,39094,2500.000000,2494.500000,2493.0,True,False,...,NaN,False,False,False,0,2_Sin_cruzar,A_2_Sin_cruzar,0.018480,True,NaN
2024-08-16 05:00:00-04:00,2500.000000,2548.300049,2498.899902,2546.199951,174111,2546.199951,2500.000000,2494.5,True,False,...,NaN,False,False,False,0,2_Sin_cruzar,A_2_Sin_cruzar,-0.001846,False,NaN
2024-08-18 17:00:00-04:00,2549.699951,2549.899902,2533.699951,2541.500000,53975,2541.500000,2546.199951,2500.0,False,True,...,2549.699951,False,False,False,0,2_Sin_cruzar,B_2_Sin_cruzar,0.000433,True,Ninguno


In [23]:
datos_h12["C1"] = datos_h12["Close"]
datos_h12["C2"] = datos_h12["Close"].shift(1)
datos_h12["C3"] = datos_h12["Close"].shift(2)

datos_h12["Escenario_A"] = (datos_h12["C1"] > datos_h12["C2"]) & (datos_h12["C1"] > datos_h12["C3"])
datos_h12["Escenario_B"] = (datos_h12["C1"] < datos_h12["C2"]) & (datos_h12["C1"] > datos_h12["C3"])

datos_h12["Es_roja"] = datos_h12["Close"] < datos_h12["Open"]
datos_h12["OURAS"] = datos_h12["Open"].where(datos_h12["Es_roja"]).ffill()

datos_h12["Sobre_OURAS_C1"] = datos_h12["C1"] > datos_h12["OURAS"]
datos_h12["Sobre_OURAS_C2"] = datos_h12["C2"] > datos_h12["OURAS"]
datos_h12["Sobre_OURAS_C3"] = datos_h12["C3"] > datos_h12["OURAS"]
datos_h12["Conteo_sobre_OURAS"] = datos_h12["Sobre_OURAS_C1"].astype(int) + datos_h12["Sobre_OURAS_C2"].astype(int) + datos_h12["Sobre_OURAS_C3"].astype(int)

datos_h12["Sub_caso"] = "N/A"
datos_h12.loc[datos_h12["Conteo_sobre_OURAS"] == 3, "Sub_caso"] = "1_Limpio"
datos_h12.loc[datos_h12["Conteo_sobre_OURAS"] == 0, "Sub_caso"] = "2_Sin_cruzar"
datos_h12.loc[(datos_h12["Conteo_sobre_OURAS"] == 1) | (datos_h12["Conteo_sobre_OURAS"] == 2), "Sub_caso"] = "3_Cruzando"

datos_h12["Combinacion"] = "Ninguno"
datos_h12.loc[datos_h12["Escenario_A"], "Combinacion"] = "A_" + datos_h12["Sub_caso"]
datos_h12.loc[datos_h12["Escenario_B"], "Combinacion"] = "B_" + datos_h12["Sub_caso"]

datos_h12["Retorno_vela_siguiente"] = datos_h12["Close"].pct_change().shift(-1)
datos_h12["Siguiente_fue_alcista"] = datos_h12["Close"].shift(-1) > datos_h12["Open"].shift(-1)

datos_h12[["Close", "Combinacion", "Retorno_vela_siguiente"]].tail(10)


,Close,Combinacion,Retorno_vela_siguiente
Datetime,,,
2026-07-31 05:00:00-04:00,4107.000000,Ninguno,0.001948
2026-08-02 17:00:00-04:00,4115.000000,Ninguno,-0.000996
2026-08-03 05:00:00-04:00,4110.899902,B_2_Sin_cruzar,-0.000024
2026-08-03 17:00:00-04:00,4110.799805,Ninguno,0.005692
2026-08-04 05:00:00-04:00,4134.200195,A_3_Cruzando,0.020681
2026-08-04 17:00:00-04:00,4219.700195,A_3_Cruzando,0.020926
2026-08-05 05:00:00-04:00,4308.000000,A_1_Limpio,0.006755
2026-08-05 17:00:00-04:00,4337.100098,A_1_Limpio,-0.008854
2026-08-06 05:00:00-04:00,4298.700195,Ninguno,0.005606


In [24]:
datos_h12_sorted = datos_h12.sort_index()
datos_w_sorted = datos_w.sort_index()

datos_h12_sorted.index = datos_h12_sorted.index.astype("datetime64[us, America/New_York]")
datos_w_sorted.index = datos_w_sorted.index.astype("datetime64[us, America/New_York]")

comparacion = pd.merge_asof(
    datos_h12_sorted,
    datos_w_sorted[["Combinacion"]].rename(columns={"Combinacion": "Combinacion_semanal"}),
    left_index=True, right_index=True,
    direction="backward"
)

comparacion[["Close", "Combinacion", "Combinacion_semanal"]].tail(10)

,Close,Combinacion,Combinacion_semanal
Datetime,,,
2026-07-31 05:00:00-04:00,4107.000000,Ninguno,Ninguno
2026-08-02 17:00:00-04:00,4115.000000,Ninguno,A_3_Cruzando
2026-08-03 05:00:00-04:00,4110.899902,B_2_Sin_cruzar,A_3_Cruzando
2026-08-03 17:00:00-04:00,4110.799805,Ninguno,A_3_Cruzando
2026-08-04 05:00:00-04:00,4134.200195,A_3_Cruzando,A_3_Cruzando
2026-08-04 17:00:00-04:00,4219.700195,A_3_Cruzando,A_3_Cruzando
2026-08-05 05:00:00-04:00,4308.000000,A_1_Limpio,A_3_Cruzando
2026-08-05 17:00:00-04:00,4337.100098,A_1_Limpio,A_3_Cruzando
2026-08-06 05:00:00-04:00,4298.700195,Ninguno,A_3_Cruzando


In [25]:
def calcular_stats(grupo):
    retornos = grupo["Retorno_vela_siguiente"].dropna()
    p_value = stats.ttest_1samp(retornos, 0)[1] if len(retornos) > 1 else None
    return pd.Series({
        "Total_casos": len(retornos),
        "Pct_siguiente_alcista": grupo["Siguiente_fue_alcista"].mean(),
        "Retorno_promedio": retornos.mean(),
        "P_value": p_value
    })

resumen_cruzado = comparacion[
    (comparacion["Combinacion_semanal"] != "Ninguno") &
    (comparacion["Combinacion"] != "Ninguno") &
    (comparacion["Combinacion_semanal"].notna())
].groupby(["Combinacion_semanal", "Combinacion"]).apply(calcular_stats)

resumen_cruzado

Total_casos  Pct_siguiente_alcista  \
Combinacion_semanal Combinacion                                          
A_1_Limpio          A_1_Limpio             18.0               0.388889   
                    A_2_Sin_cruzar          5.0               0.600000   
                    A_3_Cruzando           51.0               0.568627   
                    B_2_Sin_cruzar         12.0               0.416667   
                    B_3_Cruzando           13.0               0.461538   
A_2_Sin_cruzar      A_1_Limpio              1.0               0.000000   
                    A_2_Sin_cruzar          4.0               0.250000   
                    A_3_Cruzando           11.0               0.454545   
                    B_2_Sin_cruzar          3.0               0.666667   
                    B_3_Cruzando            2.0               0.000000   
A_3_Cruzando        A_1_Limpio             40.0               0.500000   
                    A_2_Sin_cruzar         10.0               0.700000   
                    A_3_Cruzando           65.0               0.692308   
                    B_2_Sin_cruzar         16.0               0.562500   
                    B_3_Cruzando           15.0               0.533333   
B_1_Limpio          A_1_Limpio              2.0               0.500000   
                    A_2_Sin_cruzar          1.0               0.000000   
                    A_3_Cruzando            2.0               0.500000   
                    B_2_Sin_cruzar          2.0               0.000000   
                    B_3_Cruzando            1.0               0.000000   
B_2_Sin_cruzar      A_1_Limpio              6.0               0.500000   
                    A_2_Sin_cruzar          1.0               0.000000   
                    A_3_Cruzando           15.0               0.600000   
                    B_1_Limpio              1.0               0.000000   
                    B_2_Sin_cruzar          4.0               0.750000   
                    B_3_Cruzando            2.0               0.500000   
B_3_Cruzando        A_1_Limpio              3.0               0.333333   
                    A_2_Sin_cruzar          3.0               0.333333   
                    A_3_Cruzando           16.0               0.500000   
                    B_2_Sin_cruzar          8.0               0.375000   
                    B_3_Cruzando            1.0               0.000000   

                                    Retorno_promedio   P_value  
Combinacion_semanal Combinacion                                 
A_1_Limpio          A_1_Limpio             -0.003960  0.182320  
                    A_2_Sin_cruzar          0.003899  0.276967  
                    A_3_Cruzando            0.002077  0.126916  
                    B_2_Sin_cruzar         -0.000890  0.787408  
                    B_3_Cruzando           -0.000045  0.990478  
A_2_Sin_cruzar      A_1_Limpio             -0.007315       NaN  
                    A_2_Sin_cruzar         -0.005214  0.099518  
                    A_3_Cruzando           -0.001403  0.665325  
                    B_2_Sin_cruzar         -0.000214  0.907619  
                    B_3_Cruzando           -0.006927  0.109725  
A_3_Cruzando        A_1_Limpio              0.000039  0.977738  
                    A_2_Sin_cruzar          0.002847  0.233120  
                    A_3_Cruzando            0.003298  0.004336  
                    B_2_Sin_cruzar         -0.000762  0.654206  
                    B_3_Cruzando           -0.002971  0.427054  
B_1_Limpio          A_1_Limpio              0.001597  0.891291  
                    A_2_Sin_cruzar         -0.001524       NaN  
                    A_3_Cruzando            0.001294  0.852357  
                    B_2_Sin_cruzar         -0.016292  0.455046  
                    B_3_Cruzando           -0.010225       NaN  
B_2_Sin_cruzar      A_1_Limpio              0.005376  0.281101  
                    A_2_Sin_cruzar         -0.005604       NaN  
                    A_3_Cruzando 

In [26]:
datos_h12["Escenario_A_bajista"] = (datos_h12["C1"] < datos_h12["C2"]) & (datos_h12["C1"] < datos_h12["C3"])
datos_h12["Escenario_B_bajista"] = (datos_h12["C1"] > datos_h12["C2"]) & (datos_h12["C1"] < datos_h12["C3"])

datos_h12["Es_verde"] = datos_h12["Close"] > datos_h12["Open"]
datos_h12["OUVAS"] = datos_h12["Open"].where(datos_h12["Es_verde"]).ffill()

datos_h12["Debajo_OUVAS_C1"] = datos_h12["C1"] < datos_h12["OUVAS"]
datos_h12["Debajo_OUVAS_C2"] = datos_h12["C2"] < datos_h12["OUVAS"]
datos_h12["Debajo_OUVAS_C3"] = datos_h12["C3"] < datos_h12["OUVAS"]
datos_h12["Conteo_debajo_OUVAS"] = datos_h12["Debajo_OUVAS_C1"].astype(int) + datos_h12["Debajo_OUVAS_C2"].astype(int) + datos_h12["Debajo_OUVAS_C3"].astype(int)

datos_h12["Sub_caso_bajista"] = "N/A"
datos_h12.loc[datos_h12["Conteo_debajo_OUVAS"] == 3, "Sub_caso_bajista"] = "1_Limpio"
datos_h12.loc[datos_h12["Conteo_debajo_OUVAS"] == 0, "Sub_caso_bajista"] = "2_Sin_cruzar"
datos_h12.loc[(datos_h12["Conteo_debajo_OUVAS"] == 1) | (datos_h12["Conteo_debajo_OUVAS"] == 2), "Sub_caso_bajista"] = "3_Cruzando"

datos_h12["Combinacion_bajista"] = "Ninguno"
datos_h12.loc[datos_h12["Escenario_A_bajista"], "Combinacion_bajista"] = "A_" + datos_h12["Sub_caso_bajista"]
datos_h12.loc[datos_h12["Escenario_B_bajista"], "Combinacion_bajista"] = "B_" + datos_h12["Sub_caso_bajista"]

In [27]:
def calcular_stats_bajista(grupo):
    retornos = grupo["Retorno_vela_siguiente"].dropna()
    p_value = stats.ttest_1samp(retornos, 0)[1] if len(retornos) > 1 else None
    return pd.Series({
        "Total_casos": len(retornos),
        "Pct_siguiente_bajista": 1 - grupo["Siguiente_fue_alcista"].mean(),
        "Retorno_promedio": retornos.mean(),
        "P_value": p_value
    })

In [30]:
datos_h12["Escenario_A_bajista"] = (datos_h12["C1"] < datos_h12["C2"]) & (datos_h12["C1"] < datos_h12["C3"])
datos_h12["Escenario_B_bajista"] = (datos_h12["C1"] > datos_h12["C2"]) & (datos_h12["C1"] < datos_h12["C3"])

datos_h12["Es_verde"] = datos_h12["Close"] > datos_h12["Open"]
datos_h12["OUVAS"] = datos_h12["Open"].where(datos_h12["Es_verde"]).ffill()

datos_h12["Debajo_OUVAS_C1"] = datos_h12["C1"] < datos_h12["OUVAS"]
datos_h12["Debajo_OUVAS_C2"] = datos_h12["C2"] < datos_h12["OUVAS"]
datos_h12["Debajo_OUVAS_C3"] = datos_h12["C3"] < datos_h12["OUVAS"]
datos_h12["Conteo_debajo_OUVAS"] = datos_h12["Debajo_OUVAS_C1"].astype(int) + datos_h12["Debajo_OUVAS_C2"].astype(int) + datos_h12["Debajo_OUVAS_C3"].astype(int)

datos_h12["Sub_caso_bajista"] = "N/A"
datos_h12.loc[datos_h12["Conteo_debajo_OUVAS"] == 3, "Sub_caso_bajista"] = "1_Limpio"
datos_h12.loc[datos_h12["Conteo_debajo_OUVAS"] == 0, "Sub_caso_bajista"] = "2_Sin_cruzar"
datos_h12.loc[(datos_h12["Conteo_debajo_OUVAS"] == 1) | (datos_h12["Conteo_debajo_OUVAS"] == 2), "Sub_caso_bajista"] = "3_Cruzando"

datos_h12["Combinacion_bajista"] = "Ninguno"
datos_h12.loc[datos_h12["Escenario_A_bajista"], "Combinacion_bajista"] = "A_" + datos_h12["Sub_caso_bajista"]
datos_h12.loc[datos_h12["Escenario_B_bajista"], "Combinacion_bajista"] = "B_" + datos_h12["Sub_caso_bajista"]

datos_w["Escenario_A_bajista"] = (datos_w["C1"] < datos_w["C2"]) & (datos_w["C1"] < datos_w["C3"])
datos_w["Escenario_B_bajista"] = (datos_w["C1"] > datos_w["C2"]) & (datos_w["C1"] < datos_w["C3"])
datos_w["Es_verde"] = datos_w["Close"] > datos_w["Open"]
datos_w["OUVAS"] = datos_w["Open"].where(datos_w["Es_verde"]).ffill()
datos_w["Debajo_OUVAS_C1"] = datos_w["C1"] < datos_w["OUVAS"]
datos_w["Debajo_OUVAS_C2"] = datos_w["C2"] < datos_w["OUVAS"]
datos_w["Debajo_OUVAS_C3"] = datos_w["C3"] < datos_w["OUVAS"]
datos_w["Conteo_debajo_OUVAS"] = datos_w["Debajo_OUVAS_C1"].astype(int) + datos_w["Debajo_OUVAS_C2"].astype(int) + datos_w["Debajo_OUVAS_C3"].astype(int)
datos_w["Sub_caso_bajista"] = "N/A"
datos_w.loc[datos_w["Conteo_debajo_OUVAS"] == 3, "Sub_caso_bajista"] = "1_Limpio"
datos_w.loc[datos_w["Conteo_debajo_OUVAS"] == 0, "Sub_caso_bajista"] = "2_Sin_cruzar"
datos_w.loc[(datos_w["Conteo_debajo_OUVAS"] == 1) | (datos_w["Conteo_debajo_OUVAS"] == 2), "Sub_caso_bajista"] = "3_Cruzando"
datos_w["Combinacion_bajista"] = "Ninguno"
datos_w.loc[datos_w["Escenario_A_bajista"], "Combinacion_bajista"] = "A_" + datos_w["Sub_caso_bajista"]
datos_w.loc[datos_w["Escenario_B_bajista"], "Combinacion_bajista"] = "B_" + datos_w["Sub_caso_bajista"]

datos_h12_sorted_b = datos_h12.sort_index()
datos_w_sorted_b = datos_w.sort_index()
datos_h12_sorted_b.index = datos_h12_sorted_b.index.astype("datetime64[us, America/New_York]")
datos_w_sorted_b.index = datos_w_sorted_b.index.astype("datetime64[us, America/New_York]")

comparacion_bajista = pd.merge_asof(
    datos_h12_sorted_b,
    datos_w_sorted_b[["Combinacion_bajista"]].rename(columns={"Combinacion_bajista": "Combinacion_bajista_semanal"}),
    left_index=True, right_index=True,
    direction="backward"
)

In [31]:
resumen_bajista_cruzado = comparacion_bajista[
    (comparacion_bajista["Combinacion_bajista_semanal"] != "Ninguno") &
    (comparacion_bajista["Combinacion_bajista"] != "Ninguno") &
    (comparacion_bajista["Combinacion_bajista_semanal"].notna())
].groupby(["Combinacion_bajista_semanal", "Combinacion_bajista"]).apply(calcular_stats_bajista)

resumen_bajista_cruzado

Total_casos  \
Combinacion_bajista_semanal Combinacion_bajista                
A_1_Limpio                  A_1_Limpio                   3.0   
                            A_3_Cruzando                 5.0   
                            B_2_Sin_cruzar               1.0   
                            B_3_Cruzando                 1.0   
A_2_Sin_cruzar              A_1_Limpio                   4.0   
                            A_2_Sin_cruzar               1.0   
                            A_3_Cruzando                15.0   
                            B_2_Sin_cruzar               5.0   
                            B_3_Cruzando                 1.0   
A_3_Cruzando                A_1_Limpio                   7.0   
                            A_2_Sin_cruzar              11.0   
                            A_3_Cruzando                42.0   
                            B_1_Limpio                   1.0   
                            B_2_Sin_cruzar              13.0   
                            B_3_Cruzando                15.0   
B_2_Sin_cruzar              A_1_Limpio                   2.0   
                            A_2_Sin_cruzar               1.0   
                            A_3_Cruzando                15.0   
                            B_2_Sin_cruzar               6.0   
                            B_3_Cruzando                 1.0   
B_3_Cruzando                A_1_Limpio                   1.0   
                            A_2_Sin_cruzar               3.0   
                            A_3_Cruzando                11.0   
                            B_2_Sin_cruzar               6.0   
                            B_3_Cruzando                 2.0   

                                                 Pct_siguiente_bajista  \
Combinacion_bajista_semanal Combinacion_bajista                          
A_1_Limpio                  A_1_Limpio                        0.333333   
                            A_3_Cruzando                      0.800000   
                            B_2_Sin_cruzar                    1.000000   
                            B_3_Cruzando                      1.000000   
A_2_Sin_cruzar              A_1_Limpio                        0.250000   
                            A_2_Sin_cruzar                    1.000000   
                            A_3_Cruzando                      0.533333   
                            B_2_Sin_cruzar                    0.600000   
                            B_3_Cruzando                      0.000000   
A_3_Cruzando                A_1_Limpio                        0.285714   
                            A_2_Sin_cruzar                    0.454545   
                            A_3_Cruzando                      0.404762   
                            B_1_Limpio                        0.000000   
                            B_2_Sin_cruzar                    0.538462   
                            B_3_Cruzando                      0.533333   
B_2_Sin_cruzar              A_1_Limpio                        0.000000   
                            A_2_Sin_cruzar                    0.000000   
                            A_3_Cruzando                      0.533333   
                            B_2_Sin_cruzar                    0.500000   
                            B_3_Cruzando                      1.000000   
B_3_Cruzando                A_1_Limpio                        0.000000   
                            A_2_Sin_cruzar                    0.333333   
                            A_3_Cruzando                      0.363636   
                            B_2_Sin_cruzar                    0.166667   
                            B_3_Cruzando                      0.500000   

                                                 Retorno_promedio   P_value  
Combinacion_bajista_semanal Combinacion_bajista                              
A_1_Limpio                  A_1_Limpio                   0.004985  0.544630  
                            A_3_Cruzando                -0.004451  0.290683  
         

In [ ]:
inicio_crisis = "2026-01-29"
fin_crisis = "2026-04-15"

# Verificación 1: A_3_Cruzando en H12, sin filtro semanal
datos_sin_crisis = datos_h12[(datos_h12.index < inicio_crisis) | (datos_h12.index > fin_crisis)]
casos_a3 = datos_sin_crisis[datos_sin_crisis["Combinacion"] == "A_3_Cruzando"]["Retorno_vela_siguiente"].dropna()
t_stat, p_value = stats.ttest_1samp(casos_a3, 0)
print(f"A_3_Cruzando (H12 solo) SIN crisis: n={len(casos_a3)}, retorno={casos_a3.mean():.4%}, p={p_value:.4f}")

# Verificación 2: la estrategia completa (H12 + filtro semanal)
comparacion_sin_crisis = comparacion[(comparacion.index < inicio_crisis) | (comparacion.index > fin_crisis)]
casos_filtrados = comparacion_sin_crisis[
    (comparacion_sin_crisis["Combinacion"] == "A_3_Cruzando") &
    (comparacion_sin_crisis["Combinacion_semanal"] == "A_3_Cruzando")
]["Retorno_vela_siguiente"].dropna()
t_stat2, p_value2 = stats.ttest_1samp(casos_filtrados, 0)
print(f"H12+Semanal A_3_Cruzando SIN crisis: n={len(casos_filtrados)}, retorno={casos_filtrados.mean():.4%}, p={p_value2:.4f}")

A_3_Cruzando (H12 solo) SIN crisis: n=243, retorno=0.1460%, p=0.0039
H12+Semanal A_3_Cruzando SIN crisis: n=65, retorno=0.3298%, p=0.0043


In [ ]:
import numpy as np

def analisis_completo(df_filtrado, nombre):
    retornos = df_filtrado["Retorno_vela_siguiente"].dropna()
    es_alcista = df_filtrado.loc[retornos.index, "Siguiente_fue_alcista"]

    n = len(retornos)
    win_rate = es_alcista.mean()
    retorno_promedio = retornos.mean()
    t_stat, p_value = stats.ttest_1samp(retornos, 0)

    print(f"\n{'='*55}")
    print(f"ANÁLISIS COMPLETO: {nombre}")
    print(f"{'='*55}")
    print(f"n={n} | Win rate={win_rate:.2%} | Retorno prom={retorno_promedio:.4%} | p-value={p_value:.4f}")

    np.random.seed(42)
    n_sim = 10000
    prob_perdida = 1 - win_rate
    peores_rachas = np.zeros(n_sim, dtype=int)
    for i in range(n_sim):
        resultados_sim = np.random.random(n) < prob_perdida
        racha = 0
        peor = 0
        for r in resultados_sim:
            racha = racha + 1 if r else 0
            peor = max(peor, racha)
        peores_rachas[i] = peor

    p95_racha = int(np.percentile(peores_rachas, 95))
    print(f"\nMonte Carlo rachas de pérdida ({n_sim:,} sims): promedio={peores_rachas.mean():.1f}, percentil95={p95_racha}, máximo={peores_rachas.max()}")

    print(f"\nSupervivencia de capital ante racha de {p95_racha} pérdidas seguidas:")
    for riesgo in [0.01, 0.02, 0.05, 0.10]:
        capital = (1 - riesgo) ** p95_racha
        print(f"  Riesgo {riesgo:.0%}/operación -> {capital:.1%} de capital restante")

    var_95 = np.percentile(retornos, 5)
    volatilidad = retornos.std()
    print(f"\nVaR 95% (por señal): {var_95:.4%} | Volatilidad: {volatilidad:.4%}")

    return {"n": n, "win_rate": win_rate, "retorno_promedio": retorno_promedio, "p_value": p_value,
            "p95_racha": p95_racha, "var_95": var_95, "volatilidad": volatilidad}

df1 = datos_sin_crisis[datos_sin_crisis["Combinacion"] == "A_3_Cruzando"]
resultados_h12_solo = analisis_completo(df1, "A_3_Cruzando (H12 solo, sin crisis)")

df2 = comparacion_sin_crisis[
    (comparacion_sin_crisis["Combinacion"] == "A_3_Cruzando") &
    (comparacion_sin_crisis["Combinacion_semanal"] == "A_3_Cruzando")
]
resultados_h12_semanal = analisis_completo(df2, "H12 + Semanal A_3_Cruzando (sin crisis)")


ANÁLISIS COMPLETO: A_3_Cruzando (H12 solo, sin crisis)
n=243 | Win rate=58.85% | Retorno prom=0.1460% | p-value=0.0039

Monte Carlo rachas de pérdida (10,000 sims): promedio=5.8, percentil95=8, máximo=17

Supervivencia de capital ante racha de 8 pérdidas seguidas:
  Riesgo 1%/operación -> 92.3% de capital restante
  Riesgo 2%/operación -> 85.1% de capital restante
  Riesgo 5%/operación -> 66.3% de capital restante
  Riesgo 10%/operación -> 43.0% de capital restante

VaR 95% (por señal): -1.2091% | Volatilidad: 0.7815%

ANÁLISIS COMPLETO: H12 + Semanal A_3_Cruzando (sin crisis)
n=65 | Win rate=69.23% | Retorno prom=0.3298% | p-value=0.0043

Monte Carlo rachas de pérdida (10,000 sims): promedio=3.2, percentil95=5, máximo=12

Supervivencia de capital ante racha de 5 pérdidas seguidas:
  Riesgo 1%/operación -> 95.1% de capital restante
  Riesgo 2%/operación -> 90.4% de capital restante
  Riesgo 5%/operación -> 77.4% de capital restante
  Riesgo 10%/operación -> 59.0% de capital restante



In [ ]:
def calcular_sharpe(retornos, periodos_por_año=730):
    return (retornos.mean() / retornos.std()) * np.sqrt(periodos_por_año)

sharpe_h12_solo = calcular_sharpe(df1["Retorno_vela_siguiente"].dropna())
sharpe_h12_semanal = calcular_sharpe(df2["Retorno_vela_siguiente"].dropna())

print(f"Sharpe A_3_Cruzando (H12 solo): {sharpe_h12_solo:.2f}")
print(f"Sharpe H12+Semanal: {sharpe_h12_semanal:.2f}")

Sharpe A_3_Cruzando (H12 solo): 5.05
Sharpe H12+Semanal: 9.91


In [ ]:
datos_h12["Rango"] = datos_h12["High"] - datos_h12["Low"]
datos_h12["ATR_14"] = datos_h12["Rango"].rolling(14).mean()

df1_con_atr = df1.copy()
df1_con_atr["ATR_al_momento"] = datos_h12.loc[df1_con_atr.index, "ATR_14"]

df1_con_atr["Grupo_volatilidad"] = pd.qcut(df1_con_atr["ATR_al_momento"], q=3, labels=["Baja", "Media", "Alta"])

for grupo in ["Baja", "Media", "Alta"]:
    subset = df1_con_atr[df1_con_atr["Grupo_volatilidad"] == grupo]["Retorno_vela_siguiente"].dropna()
    if len(subset) > 1:
        t_stat, p_val = stats.ttest_1samp(subset, 0)
        print(f"Volatilidad {grupo}: n={len(subset)}, retorno={subset.mean():.4%}, p={p_val:.4f}")

Volatilidad Baja: n=81, retorno=0.2539%, p=0.0000
Volatilidad Media: n=80, retorno=0.1502%, p=0.0898
Volatilidad Alta: n=80, retorno=0.0415%, p=0.7128


In [ ]:
umbral_atr_alto = df1_con_atr["ATR_al_momento"].quantile(2/3)
print(f"Regla propuesta: NO tomar la señal si el ATR_14 actual supera {umbral_atr_alto:.2f}")


Regla propuesta: NO tomar la señal si el ATR_14 actual supera 54.59


In [ ]:
df1_baja_vol = df1_con_atr[df1_con_atr["Grupo_volatilidad"] == "Baja"]

resultados_baja_vol = analisis_completo(df1_baja_vol, "A_3_Cruzando, SOLO volatilidad baja")

sharpe_baja_vol = calcular_sharpe(df1_baja_vol["Retorno_vela_siguiente"].dropna())
print(f"\nSharpe (solo volatilidad baja): {sharpe_baja_vol:.2f}")


ANÁLISIS COMPLETO: A_3_Cruzando, SOLO volatilidad baja
n=81 | Win rate=71.60% | Retorno prom=0.2539% | p-value=0.0000

Monte Carlo rachas de pérdida (10,000 sims): promedio=3.2, percentil95=5, máximo=10

Supervivencia de capital ante racha de 5 pérdidas seguidas:
  Riesgo 1%/operación -> 95.1% de capital restante
  Riesgo 2%/operación -> 90.4% de capital restante
  Riesgo 5%/operación -> 77.4% de capital restante
  Riesgo 10%/operación -> 59.0% de capital restante

VaR 95% (por señal): -0.5273% | Volatilidad: 0.4719%

Sharpe (solo volatilidad baja): 14.54


In [ ]:
win_rate_final = df1["Siguiente_fue_alcista"].mean()
n_final = len(df1["Retorno_vela_siguiente"].dropna())

np.random.seed(42)
n_sim = 10000
prob_perdida = 1 - win_rate_final
peores_rachas_final = np.zeros(n_sim, dtype=int)

for i in range(n_sim):
    resultados_sim = np.random.random(n_final) < prob_perdida
    racha = 0
    peor = 0
    for r in resultados_sim:
        racha = racha + 1 if r else 0
        peor = max(peor, racha)
    peores_rachas_final[i] = peor

p95_final = int(np.percentile(peores_rachas_final, 95))
p99_final = int(np.percentile(peores_rachas_final, 99))

print(f"A_3_Cruzando (H12 solo) — n={n_final}, win_rate={win_rate_final:.2%}")
print(f"Percentil 95 (peor racha esperada): {p95_final}")
print(f"Percentil 99 (margen extra de seguridad): {p99_final}")

A_3_Cruzando (H12 solo) — n=243, win_rate=58.85%
Percentil 95 (peor racha esperada): 8
Percentil 99 (margen extra de seguridad): 10


In [ ]:
print(f"Supervivencia de capital ante racha de {p99_final} pérdidas seguidas (percentil 99):")
for riesgo in [0.01, 0.02, 0.05, 0.10]:
    capital = (1 - riesgo) ** p99_final
    print(f"  Riesgo {riesgo:.0%}/operación -> {capital:.1%} de capital restante")

Supervivencia de capital ante racha de 10 pérdidas seguidas (percentil 99):
  Riesgo 1%/operación -> 90.4% de capital restante
  Riesgo 2%/operación -> 81.7% de capital restante
  Riesgo 5%/operación -> 59.9% de capital restante
  Riesgo 10%/operación -> 34.9% de capital restante


In [ ]:
ventana_maxima = 10

indices_señal = datos_h12.index[datos_h12["Combinacion"] == "A_3_Cruzando"]

resultados_mae_mfe = []

for fecha_entrada in indices_señal:
    posicion = datos_h12.index.get_loc(fecha_entrada)
    if posicion + ventana_maxima >= len(datos_h12):
        continue

    precio_entrada = datos_h12.loc[fecha_entrada, "Close"]
    ventana = datos_h12.iloc[posicion+1 : posicion+1+ventana_maxima]

    low_minimo = ventana["Low"].min()
    high_maximo = ventana["High"].max()

    mae = (low_minimo - precio_entrada) / precio_entrada
    mfe = (high_maximo - precio_entrada) / precio_entrada

    resultados_mae_mfe.append({
        "fecha": fecha_entrada,
        "precio_entrada": precio_entrada,
        "MAE": mae,
        "MFE": mfe
    })

tabla_mae_mfe = pd.DataFrame(resultados_mae_mfe)
tabla_mae_mfe.describe()

,precio_entrada,MAE,MFE
count,269.000000,269.000000,269.000000
mean,3669.998512,-0.020115,0.025764
std,815.591633,0.022288,0.021993
min,2526.500000,-0.184729,-0.001080
25%,2894.000000,-0.025407,0.009442
50%,3444.699951,-0.014024,0.021851
75%,4344.299805,-0.005682,0.033808
max,5408.899902,0.006165,0.143542


In [ ]:
inicio_crisis = "2026-01-29"
fin_crisis = "2026-04-15"

datos_sin_crisis_h12 = datos_h12[(datos_h12.index < inicio_crisis) | (datos_h12.index > fin_crisis)]
datos_sin_crisis_h12 = datos_sin_crisis_h12.copy()
datos_sin_crisis_h12["ATR_14"] = datos_h12["ATR_14"]

señales_con_atr = datos_sin_crisis_h12[datos_sin_crisis_h12["Combinacion"] == "A_3_Cruzando"].copy()
señales_con_atr["Grupo_volatilidad"] = pd.qcut(señales_con_atr["ATR_14"], q=3, labels=["Baja", "Media", "Alta"])

ventana_maxima = 10

for grupo in ["Baja", "Media", "Alta"]:
    fechas_grupo = señales_con_atr[señales_con_atr["Grupo_volatilidad"] == grupo].index
    maes_grupo = []
    for fecha in fechas_grupo:
        pos = datos_h12.index.get_loc(fecha)
        if pos + ventana_maxima >= len(datos_h12):
            continue
        precio_e = datos_h12.loc[fecha, "Close"]
        vent = datos_h12.iloc[pos+1 : pos+1+ventana_maxima]
        mae_g = (vent["Low"].min() - precio_e) / precio_e
        maes_grupo.append(mae_g)
    if maes_grupo:
        serie = pd.Series(maes_grupo)
        print(f"Volatilidad {grupo}: n={len(serie)}, MAE promedio={serie.mean():.4%}, percentil 85={serie.quantile(0.15):.4%}")

Volatilidad Baja: n=81, MAE promedio=-1.0414%, percentil 85=-1.8665%
Volatilidad Media: n=80, MAE promedio=-1.7488%, percentil 85=-2.8683%
Volatilidad Alta: n=78, MAE promedio=-2.7844%, percentil 85=-4.7285%


In [ ]:
for grupo in ["Baja", "Media", "Alta"]:
    fechas_grupo = señales_con_atr[señales_con_atr["Grupo_volatilidad"] == grupo].index
    mfes_grupo = []
    for fecha in fechas_grupo:
        pos = datos_h12.index.get_loc(fecha)
        if pos + ventana_maxima >= len(datos_h12):
            continue
        precio_e = datos_h12.loc[fecha, "Close"]
        vent = datos_h12.iloc[pos+1 : pos+1+ventana_maxima]
        mfe_g = (vent["High"].max() - precio_e) / precio_e
        mfes_grupo.append(mfe_g)
    if mfes_grupo:
        serie = pd.Series(mfes_grupo)
        print(f"Volatilidad {grupo}: n={len(serie)}, MFE promedio={serie.mean():.4%}, percentil 50={serie.quantile(0.50):.4%}, percentil 40={serie.quantile(0.40):.4%}")

Volatilidad Baja: n=81, MFE promedio=2.1079%, percentil 50=1.9467%, percentil 40=1.5461%
Volatilidad Media: n=80, MFE promedio=2.3404%, percentil 50=2.0692%, percentil 40=1.6602%
Volatilidad Alta: n=78, MFE promedio=3.0994%, percentil 50=2.5632%, percentil 40=2.0030%


In [ ]:
umbral_bajo = señales_con_atr["ATR_14"].quantile(1/3)
umbral_alto = señales_con_atr["ATR_14"].quantile(2/3)

def obtener_sl_tp(atr_actual):
    if atr_actual <= umbral_bajo:
        return {"SL": -0.0187, "TP": 0.0195, "grupo": "Baja"}
    elif atr_actual <= umbral_alto:
        return {"SL": -0.0287, "TP": 0.0207, "grupo": "Media"}
    else:
        return {"SL": -0.0473, "TP": 0.0256, "grupo": "Alta"}

In [ ]:
umbral_bajo = señales_con_atr["ATR_14"].quantile(1/3)
umbral_alto = señales_con_atr["ATR_14"].quantile(2/3)

tabla_sl_tp = {}
for grupo in ["Baja", "Media", "Alta"]:
    fechas_grupo = señales_con_atr[señales_con_atr["Grupo_volatilidad"] == grupo].index
    maes_grupo, mfes_grupo = [], []
    for fecha in fechas_grupo:
        pos = datos_h12.index.get_loc(fecha)
        if pos + ventana_maxima >= len(datos_h12):
            continue
        precio_e = datos_h12.loc[fecha, "Close"]
        vent = datos_h12.iloc[pos+1: pos+1+ventana_maxima]
        maes_grupo.append((vent["Low"].min() - precio_e) / precio_e)
        mfes_grupo.append((vent["High"].max() - precio_e) / precio_e)
    tabla_sl_tp[grupo] = {
        "SL": pd.Series(maes_grupo).quantile(0.15),
        "TP": pd.Series(mfes_grupo).quantile(0.50)
    }

tabla_sl_tp

{'Baja': {'SL': np.float64(-0.01866450878127947),
  'TP': np.float64(0.01946690661520386)},
 'Media': {'SL': np.float64(-0.028683072731600954),
  'TP': np.float64(0.02069163335089744)},
 'Alta': {'SL': np.float64(-0.04728452136055597),
  'TP': np.float64(0.0256322039423336)}}

In [ ]:
resultados_backtest = []

for fecha_entrada in señales_con_atr.index:
    pos = datos_h12.index.get_loc(fecha_entrada)
    if pos + ventana_maxima >= len(datos_h12):
        continue

    atr_actual = señales_con_atr.loc[fecha_entrada, "ATR_14"]
    if atr_actual <= umbral_bajo:
        grupo = tabla_sl_tp["Baja"]
    elif atr_actual <= umbral_alto:
        grupo = tabla_sl_tp["Media"]
    else:
        grupo = tabla_sl_tp["Alta"]

    nivel_sl = grupo["SL"]
    nivel_tp = grupo["TP"]
    precio_entrada = datos_h12.loc[fecha_entrada, "Close"]
    ventana = datos_h12.iloc[pos+1: pos+1+ventana_maxima]

    resultado = None
    fecha_salida = ventana.index[-1]
    retorno_operacion = (ventana["Close"].iloc[-1] - precio_entrada) / precio_entrada

    for fecha_vela, vela in ventana.iterrows():
        toco_sl = (vela["Low"] - precio_entrada) / precio_entrada <= nivel_sl
        toco_tp = (vela["High"] - precio_entrada) / precio_entrada >= nivel_tp

        if toco_sl:
            resultado = "SL"
            retorno_operacion = nivel_sl
            fecha_salida = fecha_vela
            break
        elif toco_tp:
            resultado = "TP"
            retorno_operacion = nivel_tp
            fecha_salida = fecha_vela
            break

    if resultado is None:
        resultado = "Cierre por tiempo"

    resultados_backtest.append({
        "fecha_entrada": fecha_entrada, "fecha_salida": fecha_salida,
        "resultado": resultado, "retorno": retorno_operacion
    })

tabla_backtest = pd.DataFrame(resultados_backtest)
print(tabla_backtest["resultado"].value_counts())
print(f"\nWin rate real (TP): {(tabla_backtest['resultado']=='TP').mean():.2%}")
print(f"Retorno promedio por operación: {tabla_backtest['retorno'].mean():.4%}")

resultado
TP                   119
Cierre por tiempo     89
SL                    33
Name: count, dtype: int64

Win rate real (TP): 49.38%
Retorno promedio por operación: 0.5620%


In [ ]:
tabla_backtest["gano"] = tabla_backtest["retorno"] > 0

win_rate_real = tabla_backtest["gano"].mean()
print(f"Win rate real (cualquier retorno positivo): {win_rate_real:.2%}")

print(tabla_backtest.groupby("resultado")["retorno"].agg(["count", "mean"]))

Win rate real (cualquier retorno positivo): 65.15%
                   count      mean
resultado                         
Cierre por tiempo     89 -0.002898
SL                    33 -0.030113
TP                   119  0.021899


In [ ]:
ventana_maxima_extendida = 20

resultados_backtest_v2 = []

for fecha_entrada in señales_con_atr.index:
    pos = datos_h12.index.get_loc(fecha_entrada)
    if pos + ventana_maxima_extendida >= len(datos_h12):
        continue

    atr_actual = señales_con_atr.loc[fecha_entrada, "ATR_14"]
    if atr_actual <= umbral_bajo:
        grupo = tabla_sl_tp["Baja"]
    elif atr_actual <= umbral_alto:
        grupo = tabla_sl_tp["Media"]
    else:
        grupo = tabla_sl_tp["Alta"]

    nivel_sl, nivel_tp = grupo["SL"], grupo["TP"]
    precio_entrada = datos_h12.loc[fecha_entrada, "Close"]
    ventana = datos_h12.iloc[pos+1: pos+1+ventana_maxima_extendida]

    resultado = None
    retorno_operacion = (ventana["Close"].iloc[-1] - precio_entrada) / precio_entrada

    for fecha_vela, vela in ventana.iterrows():
        if (vela["Low"] - precio_entrada) / precio_entrada <= nivel_sl:
            resultado, retorno_operacion = "SL", nivel_sl
            break
        elif (vela["High"] - precio_entrada) / precio_entrada >= nivel_tp:
            resultado, retorno_operacion = "TP", nivel_tp
            break

    if resultado is None:
        resultado = "Cierre por tiempo"

    resultados_backtest_v2.append({"resultado": resultado, "retorno": retorno_operacion})

tabla_v2 = pd.DataFrame(resultados_backtest_v2)
print(tabla_v2["resultado"].value_counts())
print(f"Win rate real: {(tabla_v2['retorno']>0).mean():.2%}")
print(f"Retorno promedio: {tabla_v2['retorno'].mean():.4%}")

resultado
TP                   164
SL                    50
Cierre por tiempo     23
Name: count, dtype: int64
Win rate real: 73.84%
Retorno promedio: 0.8465%


In [ ]:
ventana_sin_limite = 60

resultados_backtest_v3 = []

for fecha_entrada in señales_con_atr.index:
    pos = datos_h12.index.get_loc(fecha_entrada)
    if pos + ventana_sin_limite >= len(datos_h12):
        continue

    atr_actual = señales_con_atr.loc[fecha_entrada, "ATR_14"]
    if atr_actual <= umbral_bajo:
        grupo = tabla_sl_tp["Baja"]
    elif atr_actual <= umbral_alto:
        grupo = tabla_sl_tp["Media"]
    else:
        grupo = tabla_sl_tp["Alta"]

    nivel_sl, nivel_tp = grupo["SL"], grupo["TP"]
    precio_entrada = datos_h12.loc[fecha_entrada, "Close"]
    ventana = datos_h12.iloc[pos+1: pos+1+ventana_sin_limite]

    resultado = None
    retorno_operacion = None

    for fecha_vela, vela in ventana.iterrows():
        if (vela["Low"] - precio_entrada) / precio_entrada <= nivel_sl:
            resultado, retorno_operacion = "SL", nivel_sl
            break
        elif (vela["High"] - precio_entrada) / precio_entrada >= nivel_tp:
            resultado, retorno_operacion = "TP", nivel_tp
            break

    if resultado is None:
        resultado = "Sin resolver en 60 velas"
        retorno_operacion = (ventana["Close"].iloc[-1] - precio_entrada) / precio_entrada

    resultados_backtest_v3.append({"resultado": resultado, "retorno": retorno_operacion})

tabla_v3 = pd.DataFrame(resultados_backtest_v3)
print(tabla_v3["resultado"].value_counts())
print(f"Win rate real: {(tabla_v3['retorno']>0).mean():.2%}")
print(f"Retorno promedio: {tabla_v3['retorno'].mean():.4%}")

resultado
TP    172
SL     57
Name: count, dtype: int64
Win rate real: 75.11%
Retorno promedio: 0.8535%


In [ ]:
ganancia_promedio_cuando_gano = tabla_v3[tabla_v3["resultado"] == "TP"]["retorno"].mean()
perdida_promedio_cuando_pierdo = tabla_v3[tabla_v3["resultado"] == "SL"]["retorno"].mean()

print(f"Cuando gano (TP): {ganancia_promedio_cuando_gano:.4%}")
print(f"Cuando pierdo (SL): {perdida_promedio_cuando_pierdo:.4%}")
print(f"Payoff Ratio: {abs(ganancia_promedio_cuando_gano / perdida_promedio_cuando_pierdo):.2f}")

Cuando gano (TP): 2.1701%
Cuando pierdo (SL): -3.1193%
Payoff Ratio: 0.70


In [ ]:
from scipy import stats
import numpy as np

retornos_backtest = tabla_v3["retorno"]

t_stat, p_value_backtest = stats.ttest_1samp(retornos_backtest, 0)

años_cubiertos = (señales_con_atr.index.max() - señales_con_atr.index.min()).days / 365
operaciones_por_año = len(tabla_v3) / años_cubiertos

sharpe_backtest = (retornos_backtest.mean() / retornos_backtest.std()) * np.sqrt(operaciones_por_año)

var_95_backtest = np.percentile(retornos_backtest, 5)

capital_backtest = (1 + retornos_backtest).cumprod()
maximo_backtest = capital_backtest.cummax()
drawdown_backtest = (capital_backtest / maximo_backtest) - 1
drawdown_maximo_backtest = drawdown_backtest.min()

print(f"P-value: {p_value_backtest:.6f}")
print(f"Sharpe (anualizado, ~{operaciones_por_año:.0f} operaciones/año): {sharpe_backtest:.2f}")
print(f"VaR 95%: {var_95_backtest:.4%}")
print(f"Drawdown máximo (curva de operaciones compuestas): {drawdown_maximo_backtest:.2%}")

P-value: 0.000000
Sharpe (anualizado, ~116 operaciones/año): 3.87
VaR 95%: -4.7285%
Drawdown máximo (curva de operaciones compuestas): -45.12%


In [ ]:
riesgo_por_operacion = 0.01

r_multiples = []
for idx, fila in tabla_v3.iterrows():
    for grupo_nombre, niveles in tabla_sl_tp.items():
        if abs(fila["retorno"] - niveles["SL"]) < 0.0001:
            r_multiples.append(-1.0)
            break
        elif abs(fila["retorno"] - niveles["TP"]) < 0.0001:
            r_multiples.append(niveles["TP"] / abs(niveles["SL"]))
            break

tabla_v3["R_multiple"] = r_multiples

capital_real = (1 + riesgo_por_operacion * tabla_v3["R_multiple"]).cumprod()
maximo_real = capital_real.cummax()
drawdown_real_correcto = ((capital_real / maximo_real) - 1).min()

print(f"Drawdown máximo con riesgo real (1% por operación): {drawdown_real_correcto:.2%}")


Drawdown máximo con riesgo real (1% por operación): -11.64%


In [ ]:
win_rate_backtest = (tabla_v3["retorno"] > 0).mean()
n_operaciones = len(tabla_v3)

np.random.seed(42)
n_sim = 10000
prob_perdida = 1 - win_rate_backtest
peores_rachas_backtest = np.zeros(n_sim, dtype=int)

for i in range(n_sim):
    resultados_sim = np.random.random(n_operaciones) < prob_perdida
    racha, peor = 0, 0
    for r in resultados_sim:
        racha = racha + 1 if r else 0
        peor = max(peor, racha)
    peores_rachas_backtest[i] = peor

p95_bt = int(np.percentile(peores_rachas_backtest, 95))
p99_bt = int(np.percentile(peores_rachas_backtest, 99))

riesgo = 0.01
capital_p95 = (1 - riesgo) ** p95_bt
capital_p99 = (1 - riesgo) ** p99_bt

print(f"Monte Carlo (win rate real 75.11%, n={n_operaciones}):")
print(f"  Percentil 95: {p95_bt} pérdidas seguidas -> drawdown implícito: {(capital_p95-1):.2%}")
print(f"  Percentil 99: {p99_bt} pérdidas seguidas -> drawdown implícito: {(capital_p99-1):.2%}")
print(f"\nDrawdown REAL observado en el backtest: -11.64%")

capital_final = capital_real.iloc[-1]
print(f"\nCon 1% de riesgo por operación, capital final tras {n_operaciones} señales: {capital_final:.4f}")
print(f"Ganancia total del periodo: {(capital_final-1):.2%}")

capital_inicial_ejemplo = 1000
print(f"Ejemplo con $1,000 iniciales: terminarías con ${capital_inicial_ejemplo * capital_final:,.2f}")

Monte Carlo (win rate real 75.11%, n=229):
  Percentil 95: 5 pérdidas seguidas -> drawdown implícito: -4.90%
  Percentil 99: 6 pérdidas seguidas -> drawdown implícito: -5.85%

Drawdown REAL observado en el backtest: -11.64%

Con 1% de riesgo por operación, capital final tras 229 señales: 2.1504
Ganancia total del periodo: 115.04%
Ejemplo con $1,000 iniciales: terminarías con $2,150.39


In [ ]:
np.random.seed(42)
n_sim = 10000
drawdowns_bootstrap = []

for i in range(n_sim):
    muestra = np.random.choice(retornos_backtest.values, size=len(retornos_backtest), replace=True)
    r_muestra = np.where(muestra > 0,
                          riesgo_por_operacion * (tabla_v3["R_multiple"].mean()),
                          -riesgo_por_operacion)
    capital_sim = (1 + r_muestra).cumprod()
    dd_sim = ((capital_sim / np.maximum.accumulate(capital_sim)) - 1).min()
    drawdowns_bootstrap.append(dd_sim)

drawdowns_bootstrap = np.array(drawdowns_bootstrap)
print(f"Drawdown Bootstrap - percentil 95: {np.percentile(drawdowns_bootstrap, 5):.2%}")
print(f"Drawdown Bootstrap - percentil 99: {np.percentile(drawdowns_bootstrap, 1):.2%}")
print(f"Drawdown REAL observado: -11.64%")

Drawdown Bootstrap - percentil 95: -17.22%
Drawdown Bootstrap - percentil 99: -21.25%
Drawdown REAL observado: -11.64%


In [ ]:
capital_inicial = 1000
capital = capital_inicial

print(f"Capital inicial: ${capital:,.2f}")
for i, r_multiple in enumerate(tabla_v3["R_multiple"].head(5), 1):
    riesgo_dolares = capital * 0.01
    resultado_dolares = riesgo_dolares * r_multiple
    capital = capital + resultado_dolares
    print(f"Operación {i}: arriesgué ${riesgo_dolares:,.2f} (1% de ${capital-resultado_dolares:,.2f}) -> resultado ${resultado_dolares:,.2f} -> capital nuevo: ${capital:,.2f}")

Capital inicial: $1,000.00
Operación 1: arriesgué $10.00 (1% de $1,000.00) -> resultado $5.42 -> capital nuevo: $1,005.42
Operación 2: arriesgué $10.05 (1% de $1,005.42) -> resultado $5.45 -> capital nuevo: $1,010.87
Operación 3: arriesgué $10.11 (1% de $1,010.87) -> resultado $10.54 -> capital nuevo: $1,021.41
Operación 4: arriesgué $10.21 (1% de $1,021.41) -> resultado $-10.21 -> capital nuevo: $1,011.20
Operación 5: arriesgué $10.11 (1% de $1,011.20) -> resultado $-10.11 -> capital nuevo: $1,001.09


In [ ]:
def calcular_tamaño_posicion(capital_actual, riesgo_pct, distancia_sl_pct):
    dinero_a_arriesgar = capital_actual * riesgo_pct
    tamaño_posicion_dolares = dinero_a_arriesgar / abs(distancia_sl_pct)
    return tamaño_posicion_dolares

ejemplo = calcular_tamaño_posicion(capital_actual=1000, riesgo_pct=0.01, distancia_sl_pct=0.0187)
print(f"Con $1,000 de capital y SL de 1.87%, tu posición debería ser de ${ejemplo:,.2f}")

ejemplo2 = calcular_tamaño_posicion(capital_actual=2150, riesgo_pct=0.01, distancia_sl_pct=0.0187)
print(f"Con $2,150 de capital (ya creciste) y el mismo SL de 1.87%, tu posición debería ser de ${ejemplo2:,.2f}")

Con $1,000 de capital y SL de 1.87%, tu posición debería ser de $534.76
Con $2,150 de capital (ya creciste) y el mismo SL de 1.87%, tu posición debería ser de $1,149.73


In [ ]:
capital_inicial = 1000
capital_dolares = capital_inicial * capital_real

print(f"Capital inicial: ${capital_inicial:,.2f}")
print(f"Número total de operaciones: {len(capital_real)}")
print(f"\nCapital en distintos puntos del camino:")
print(f"  Después de 50 operaciones: ${capital_dolares.iloc[49]:,.2f}")
print(f"  Después de 100 operaciones: ${capital_dolares.iloc[99]:,.2f}")
print(f"  Después de 150 operaciones: ${capital_dolares.iloc[149]:,.2f}")
print(f"  Después de 200 operaciones: ${capital_dolares.iloc[199]:,.2f}")
print(f"  Capital FINAL (las {len(capital_real)} operaciones): ${capital_dolares.iloc[-1]:,.2f}")

ganancia_total = capital_dolares.iloc[-1] - capital_inicial
retorno_total_pct = (capital_dolares.iloc[-1] / capital_inicial - 1)
print(f"\nGanancia total: ${ganancia_total:,.2f}")
print(f"Retorno total: {retorno_total_pct:.2%}")

Capital inicial: $1,000.00
Número total de operaciones: 229

Capital en distintos puntos del camino:
  Después de 50 operaciones: $1,214.34
  Después de 100 operaciones: $1,513.63
  Después de 150 operaciones: $1,902.12
  Después de 200 operaciones: $2,305.67
  Capital FINAL (las 229 operaciones): $2,150.39

Ganancia total: $1,150.39
Retorno total: 115.04%


In [ ]:
capital_inicial = 1000
capital_dolares = capital_inicial * capital_real

print(f"Capital inicial: ${capital_inicial:,.2f}")
print(f"Número total de operaciones: {len(capital_real)}")
print(f"\nCapital en distintos puntos del camino:")
print(f"  Después de 50 operaciones: ${capital_dolares.iloc[49]:,.2f}")
print(f"  Después de 100 operaciones: ${capital_dolares.iloc[99]:,.2f}")
print(f"  Después de 150 operaciones: ${capital_dolares.iloc[149]:,.2f}")
print(f"  Después de 200 operaciones: ${capital_dolares.iloc[199]:,.2f}")
print(f"  Capital FINAL (las {len(capital_real)} operaciones): ${capital_dolares.iloc[-1]:,.2f}")

ganancia_total = capital_dolares.iloc[-1] - capital_inicial
retorno_total_pct = (capital_dolares.iloc[-1] / capital_inicial - 1)
print(f"\nGanancia total: ${ganancia_total:,.2f}")
print(f"Retorno total: {retorno_total_pct:.2%}")

Capital inicial: $1,000.00
Número total de operaciones: 229

Capital en distintos puntos del camino:
  Después de 50 operaciones: $1,214.34
  Después de 100 operaciones: $1,513.63
  Después de 150 operaciones: $1,902.12
  Después de 200 operaciones: $2,305.67
  Capital FINAL (las 229 operaciones): $2,150.39

Ganancia total: $1,150.39
Retorno total: 115.04%


In [ ]:
capital_inicial = 1000

# Versión A: interés compuesto (1% del capital ACTUAL, ya calculado como capital_real)
capital_compuesto = capital_inicial * capital_real.iloc[-1]

# Versión B: riesgo fijo (1% del capital ORIGINAL, siempre $10 fijos)
riesgo_fijo_dolares = capital_inicial * 0.01
ganancia_total_fijo = (riesgo_fijo_dolares * tabla_v3["R_multiple"]).sum()
capital_fijo = capital_inicial + ganancia_total_fijo

print(f"Capital inicial: ${capital_inicial:,.2f}\n")
print(f"VERSIÓN A - Interés compuesto (1% del capital actual, va creciendo):")
print(f"  Capital final: ${capital_compuesto:,.2f}")
print(f"  Ganancia: {(capital_compuesto/capital_inicial - 1):.2%}\n")
print(f"VERSIÓN B - Riesgo fijo (siempre 1% de los $1,000 originales, $10 fijos):")
print(f"  Capital final: ${capital_fijo:,.2f}")
print(f"  Ganancia: {(capital_fijo/capital_inicial - 1):.2%}")


Capital inicial: $1,000.00

VERSIÓN A - Interés compuesto (1% del capital actual, va creciendo):
  Capital final: $2,150.39
  Ganancia: 115.04%

VERSIÓN B - Riesgo fijo (siempre 1% de los $1,000 originales, $10 fijos):
  Capital final: $1,774.10
  Ganancia: 77.41%


In [ ]:
riesgo_ejemplo = 0.10  # SOLO PARA VER EL EFECTO, NO PARA OPERAR ASÍ

capital_10pct = (1 + riesgo_ejemplo * tabla_v3["R_multiple"]).cumprod()
capital_dolares_10pct = 1000 * capital_10pct

print(f"Con 10% de riesgo (ejemplo ilustrativo, NO recomendado): ${capital_dolares_10pct.iloc[-1]:,.2f}")

Con 10% de riesgo (ejemplo ilustrativo, NO recomendado): $997,500.81


In [ ]:
maximo_10pct = capital_10pct.cummax()
drawdown_10pct = (capital_10pct / maximo_10pct) - 1
peor_momento_10pct = drawdown_10pct.min()

print(f"Peor drawdown con 10% de riesgo: {peor_momento_10pct:.2%}")
print(f"Fecha del peor momento: {drawdown_10pct.idxmin()}")

Peor drawdown con 10% de riesgo: -73.19%
Fecha del peor momento: 227


In [ ]:
capital_dolares_10pct = 1000 * capital_10pct

print("Capital en las últimas operaciones:")
print(capital_dolares_10pct.tail(10))

print(f"\nPico máximo alcanzado ANTES de la operación 227:")
print(f"{maximo_10pct.iloc[226]:,.2f}")

print(f"\nCapital exacto en la operación 227 (el peor momento):")
print(f"{capital_dolares_10pct.iloc[226]:,.2f}")

Capital en las últimas operaciones:
219    1.602060e+06
220    1.441854e+06
221    1.297668e+06
222    1.167902e+06
223    1.051111e+06
224    1.108091e+06
225    1.168158e+06
226    1.051343e+06
227    9.462083e+05
228    9.975008e+05
Name: R_multiple, dtype: float64

Pico máximo alcanzado ANTES de la operación 227:
3,529.53

Capital exacto en la operación 227 (el peor momento):
1,051,342.59


In [ ]:
pico_real_dolares = 1000 * maximo_10pct.iloc[226]
print(f"Pico máximo alcanzado (en dólares): ${pico_real_dolares:,.2f}")

Pico máximo alcanzado (en dólares): $3,529,528.31


In [ ]:
maximo_1pct_dolares = 1000 * capital_real.cummax()
pico_1pct = maximo_1pct_dolares.max()
capital_final_1pct = 1000 * capital_real.iloc[-1]

print(f"Pico máximo (1% de riesgo): ${pico_1pct:,.2f}")
print(f"Capital final: ${capital_final_1pct:,.2f}")
print(f"Diferencia entre pico y final: {(capital_final_1pct/pico_1pct - 1):.2%}")

Pico máximo (1% de riesgo): $2,420.48
Capital final: $2,150.39
Diferencia entre pico y final: -11.16%


In [ ]:
señales_ordenadas = señales_con_atr.index.sort_values()

solapamientos = 0
detalle_solapamientos = []

for i, fecha_señal in enumerate(señales_ordenadas):
    pos = datos_h12.index.get_loc(fecha_señal)
    if pos + 1 >= len(datos_h12):
        continue

    atr_actual = señales_con_atr.loc[fecha_señal, "ATR_14"]
    if atr_actual <= umbral_bajo:
        grupo = tabla_sl_tp["Baja"]
    elif atr_actual <= umbral_alto:
        grupo = tabla_sl_tp["Media"]
    else:
        grupo = tabla_sl_tp["Alta"]

    precio_e = datos_h12.loc[fecha_señal, "Close"]
    ventana = datos_h12.iloc[pos+1:]

    fecha_cierre = None
    for fecha_v, vela in ventana.iterrows():
        if (vela["Low"]-precio_e)/precio_e <= grupo["SL"] or (vela["High"]-precio_e)/precio_e >= grupo["TP"]:
            fecha_cierre = fecha_v
            break

    if fecha_cierre is not None:
        siguientes_señales = señales_ordenadas[(señales_ordenadas > fecha_señal) & (señales_ordenadas <= fecha_cierre)]
        if len(siguientes_señales) > 0:
            solapamientos += 1

print(f"De {len(señales_ordenadas)} señales, {solapamientos} se solaparon con al menos otra operación abierta ({solapamientos/len(señales_ordenadas):.1%})")

De 243 señales, 209 se solaparon con al menos otra operación abierta (86.0%)


In [ ]:
eventos = []
for i, fecha_señal in enumerate(señales_ordenadas):
    pos = datos_h12.index.get_loc(fecha_señal)
    if pos + 1 >= len(datos_h12):
        continue
    atr_actual = señales_con_atr.loc[fecha_señal, "ATR_14"]
    if atr_actual <= umbral_bajo:
        grupo = tabla_sl_tp["Baja"]
    elif atr_actual <= umbral_alto:
        grupo = tabla_sl_tp["Media"]
    else:
        grupo = tabla_sl_tp["Alta"]
    precio_e = datos_h12.loc[fecha_señal, "Close"]
    ventana = datos_h12.iloc[pos+1:]
    fecha_cierre = ventana.index[-1]
    for fecha_v, vela in ventana.iterrows():
        if (vela["Low"]-precio_e)/precio_e <= grupo["SL"] or (vela["High"]-precio_e)/precio_e >= grupo["TP"]:
            fecha_cierre = fecha_v
            break
    eventos.append((fecha_señal, 1))
    eventos.append((fecha_cierre, -1))

eventos.sort()
posiciones_abiertas, maximo_simultaneo = 0, 0
for fecha, cambio in eventos:
    posiciones_abiertas += cambio
    maximo_simultaneo = max(maximo_simultaneo, posiciones_abiertas)

print(f"Máximo de posiciones abiertas simultáneamente: {maximo_simultaneo}")
print(f"Riesgo real máximo simultáneo (a 1% cada una): {maximo_simultaneo}%")

Máximo de posiciones abiertas simultáneamente: 9
Riesgo real máximo simultáneo (a 1% cada una): 9%


In [ ]:
resultados_salida_tendencial = []

for fecha_entrada in señales_con_atr.index:
    pos = datos_h12.index.get_loc(fecha_entrada)
    if pos + 1 >= len(datos_h12):
        continue

    precio_entrada = datos_h12.loc[fecha_entrada, "Close"]
    ventana = datos_h12.iloc[pos+1:]

    mfe_maximo_idealizado = 0
    fecha_reversion = None
    precio_en_reversion = None

    for fecha_v, vela in ventana.iterrows():
        mfe_actual = (vela["High"] - precio_entrada) / precio_entrada
        mfe_maximo_idealizado = max(mfe_maximo_idealizado, mfe_actual)

        if vela["Escenario_A_bajista"]:
            fecha_reversion = fecha_v
            precio_en_reversion = vela["Close"]
            break

    if fecha_reversion is None:
        continue

    retorno_realista = (precio_en_reversion - precio_entrada) / precio_entrada

    resultados_salida_tendencial.append({
        "fecha_entrada": fecha_entrada,
        "MFE_idealizado_imposible_de_capturar_exacto": mfe_maximo_idealizado,
        "retorno_realista_al_detectar_reversion": retorno_realista
    })

tabla_salida_tendencial = pd.DataFrame(resultados_salida_tendencial)
print(f"Casos con reversión detectada: {len(tabla_salida_tendencial)} de {len(señales_con_atr)}")
tabla_salida_tendencial.describe()

Casos con reversión detectada: 243 de 243


,MFE_idealizado_imposible_de_capturar_exacto,retorno_realista_al_detectar_reversion
count,243.000000,243.000000
mean,0.019037,0.003457
std,0.026624,0.017850
min,0.000000,-0.053906
25%,0.003607,-0.006342
50%,0.011803,-0.001223
75%,0.022000,0.009322
max,0.204573,0.101430


In [ ]:
mfes_hasta_reversion = []

for fecha_entrada in señales_con_atr.index:
    pos = datos_h12.index.get_loc(fecha_entrada)
    if pos + 1 >= len(datos_h12):
        continue

    precio_entrada = datos_h12.loc[fecha_entrada, "Close"]
    ventana = datos_h12.iloc[pos+1:]

    mfe_de_esta_operacion = 0
    encontro_reversion = False

    for fecha_v, vela in ventana.iterrows():
        mfe_actual = (vela["High"] - precio_entrada) / precio_entrada
        mfe_de_esta_operacion = max(mfe_de_esta_operacion, mfe_actual)

        if vela["Escenario_A_bajista"]:
            encontro_reversion = True
            break

    if encontro_reversion:
        mfes_hasta_reversion.append(mfe_de_esta_operacion)

serie_mfe_reversion = pd.Series(mfes_hasta_reversion)
print(f"n={len(serie_mfe_reversion)}")
print(f"MFE promedio (hasta reversión): {serie_mfe_reversion.mean():.4%}")
print(f"MFE percentil 50: {serie_mfe_reversion.quantile(0.50):.4%}")
print(f"MFE percentil 40: {serie_mfe_reversion.quantile(0.40):.4%}")

n=243
MFE promedio (hasta reversión): 1.9037%
MFE percentil 50: 1.1803%
MFE percentil 40: 0.8224%


In [ ]:
tp_ambicioso = serie_mfe_reversion.quantile(0.85)
print(f"TP ambicioso (percentil 85): {tp_ambicioso:.4%}\n")

resultados_combinado = []

for fecha_entrada in señales_con_atr.index:
    pos = datos_h12.index.get_loc(fecha_entrada)
    if pos + 1 >= len(datos_h12):
        continue

    atr_actual = señales_con_atr.loc[fecha_entrada, "ATR_14"]
    if atr_actual <= umbral_bajo:
        nivel_sl = tabla_sl_tp["Baja"]["SL"]
    elif atr_actual <= umbral_alto:
        nivel_sl = tabla_sl_tp["Media"]["SL"]
    else:
        nivel_sl = tabla_sl_tp["Alta"]["SL"]

    precio_entrada = datos_h12.loc[fecha_entrada, "Close"]
    ventana = datos_h12.iloc[pos+1:]

    resultado, retorno_op = None, None

    for fecha_v, vela in ventana.iterrows():
        ret_low = (vela["Low"] - precio_entrada) / precio_entrada
        ret_high = (vela["High"] - precio_entrada) / precio_entrada

        if ret_low <= nivel_sl:
            resultado, retorno_op = "SL", nivel_sl
            break
        elif ret_high >= tp_ambicioso:
            resultado, retorno_op = "TP_ambicioso", tp_ambicioso
            break
        elif vela["Escenario_A_bajista"]:
            resultado = "Salida_por_reversion"
            retorno_op = (vela["Close"] - precio_entrada) / precio_entrada
            break

    if resultado:
        resultados_combinado.append({"resultado": resultado, "retorno": retorno_op})

tabla_combinado = pd.DataFrame(resultados_combinado)
print(tabla_combinado["resultado"].value_counts())
print(f"\nWin rate: {(tabla_combinado['retorno']>0).mean():.2%}")
print(f"Retorno promedio: {tabla_combinado['retorno'].mean():.4%}")

TP ambicioso (percentil 85): 3.3939%

resultado
Salida_por_reversion    204
TP_ambicioso             37
SL                        2
Name: count, dtype: int64

Win rate: 46.91%
Retorno promedio: 0.3727%


In [ ]:
resultados_combinado_v2 = []

for fecha_entrada in señales_con_atr.index:
    pos = datos_h12.index.get_loc(fecha_entrada)
    if pos + 1 >= len(datos_h12):
        continue

    atr_actual = señales_con_atr.loc[fecha_entrada, "ATR_14"]
    if atr_actual <= umbral_bajo:
        nivel_sl = tabla_sl_tp["Baja"]["SL"]
    elif atr_actual <= umbral_alto:
        nivel_sl = tabla_sl_tp["Media"]["SL"]
    else:
        nivel_sl = tabla_sl_tp["Alta"]["SL"]

    precio_entrada = datos_h12.loc[fecha_entrada, "Close"]
    ventana = datos_h12.iloc[pos+1:]

    resultado, retorno_op = None, None

    for fecha_v, vela in ventana.iterrows():
        ret_low = (vela["Low"] - precio_entrada) / precio_entrada
        ret_high = (vela["High"] - precio_entrada) / precio_entrada

        reversion_confirmada = vela["Escenario_A_bajista"] and vela["Sub_caso_bajista"] == "3_Cruzando"

        if ret_low <= nivel_sl:
            resultado, retorno_op = "SL", nivel_sl
            break
        elif ret_high >= tp_ambicioso:
            resultado, retorno_op = "TP_ambicioso", tp_ambicioso
            break
        elif reversion_confirmada:
            resultado = "Salida_por_reversion_confirmada"
            retorno_op = (vela["Close"] - precio_entrada) / precio_entrada
            break

    if resultado:
        resultados_combinado_v2.append({"resultado": resultado, "retorno": retorno_op})

tabla_v2_reversion = pd.DataFrame(resultados_combinado_v2)
print(tabla_v2_reversion["resultado"].value_counts())
print(f"\nWin rate: {(tabla_v2_reversion['retorno']>0).mean():.2%}")
print(f"Retorno promedio: {tabla_v2_reversion['retorno'].mean():.4%}")

resultado
Salida_por_reversion_confirmada    201
TP_ambicioso                        38
SL                                   4
Name: count, dtype: int64

Win rate: 47.74%
Retorno promedio: 0.3571%


In [ ]:
señales_ordenadas = señales_con_atr.sort_index()
resultados_unica = []
fecha_libre_desde = señales_ordenadas.index.min() - pd.Timedelta(days=1)

In [ ]:
for fecha_entrada in señales_ordenadas.index:
    if fecha_entrada <= fecha_libre_desde:
        continue
    pos = datos_h12.index.get_loc(fecha_entrada)
    if pos + 1 >= len(datos_h12):
        continue
    atr_actual = señales_con_atr.loc[fecha_entrada, "ATR_14"]
    if atr_actual <= umbral_bajo:
        grupo = tabla_sl_tp["Baja"]
    elif atr_actual <= umbral_alto:
        grupo = tabla_sl_tp["Media"]
    else:
        grupo = tabla_sl_tp["Alta"]
    nivel_sl = grupo["SL"]
    nivel_tp = grupo["TP"]
    precio_entrada = datos_h12.loc[fecha_entrada, "Close"]
    ventana = datos_h12.iloc[pos+1:]
    resultado = None
    retorno_op = None
    fecha_salida = None
    for fecha_v, vela in ventana.iterrows():
        ret_low = (vela["Low"] - precio_entrada) / precio_entrada
        ret_high = (vela["High"] - precio_entrada) / precio_entrada
        if ret_low <= nivel_sl:
            resultado = "SL"
            retorno_op = nivel_sl
            fecha_salida = fecha_v
            break
        elif ret_high >= nivel_tp:
            resultado = "TP"
            retorno_op = nivel_tp
            fecha_salida = fecha_v
            break
    if resultado is None:
        continue
    fecha_libre_desde = fecha_salida
    resultados_unica.append({"fecha_entrada": fecha_entrada, "fecha_salida": fecha_salida, "resultado": resultado, "retorno": retorno_op})

In [ ]:
tabla_unica = pd.DataFrame(resultados_unica)
dias_cubiertos = (tabla_unica["fecha_entrada"].max() - tabla_unica["fecha_entrada"].min()).days
años_cubiertos = dias_cubiertos / 365
print(f"Señales originales: {len(señales_con_atr)}")
print(f"Operaciones ejecutadas: {len(tabla_unica)}")
print(tabla_unica["resultado"].value_counts())
print(f"Win rate: {(tabla_unica['retorno']>0).mean():.2%}")
print(f"Retorno promedio: {tabla_unica['retorno'].mean():.4%}")
print(f"Frecuencia: {len(tabla_unica)/años_cubiertos:.1f} operaciones/año")

Señales originales: 243
Operaciones ejecutadas: 60
resultado
TP    44
SL    16
Name: count, dtype: int64
Win rate: 73.33%
Retorno promedio: 0.7464%
Frecuencia: 31.7 operaciones/año


In [ ]:
señales_ordenadas = señales_con_atr.sort_index()
resultados_unica = []
fecha_libre_desde = señales_ordenadas.index.min() - pd.Timedelta(days=1)

for fecha_entrada in señales_ordenadas.index:
    if fecha_entrada <= fecha_libre_desde:
        continue
    pos = datos_h12.index.get_loc(fecha_entrada)
    if pos + 1 >= len(datos_h12):
        continue
    atr_actual = señales_con_atr.loc[fecha_entrada, "ATR_14"]
    if atr_actual <= umbral_bajo:
        grupo = tabla_sl_tp["Baja"]
    elif atr_actual <= umbral_alto:
        grupo = tabla_sl_tp["Media"]
    else:
        grupo = tabla_sl_tp["Alta"]
    nivel_sl = grupo["SL"]
    nivel_tp = grupo["TP"]
    precio_entrada = datos_h12.loc[fecha_entrada, "Close"]
    ventana = datos_h12.iloc[pos+1:]
    resultado = None
    fecha_salida = None
    for fecha_v, vela in ventana.iterrows():
        if (vela["Low"] - precio_entrada) / precio_entrada <= nivel_sl:
            resultado = "SL"
            fecha_salida = fecha_v
            break
        elif (vela["High"] - precio_entrada) / precio_entrada >= nivel_tp:
            resultado = "TP"
            fecha_salida = fecha_v
            break
    if resultado is None:
        continue
    fecha_libre_desde = fecha_salida
    resultados_unica.append({"fecha_entrada": fecha_entrada, "resultado": resultado})

print(f"Señales originales: {len(señales_con_atr)}")
print(f"Operaciones ejecutadas (esperando cierre): {len(resultados_unica)}")
print(f"Señales descartadas por solapamiento: {len(señales_con_atr) - len(resultados_unica)}")
print(f"Porcentaje descartado: {(len(señales_con_atr) - len(resultados_unica))/len(señales_con_atr):.1%}")

Señales originales: 243
Operaciones ejecutadas (esperando cierre): 60
Señales descartadas por solapamiento: 183
Porcentaje descartado: 75.3%


In [ ]:
señales_ordenadas = señales_con_atr.sort_index()
limite_posiciones = 3

posiciones_abiertas_actual = []
resultados_max3 = []

for fecha_entrada in señales_ordenadas.index:
    posiciones_abiertas_actual = [f for f in posiciones_abiertas_actual if f > fecha_entrada]

    if len(posiciones_abiertas_actual) >= limite_posiciones:
        continue

    pos = datos_h12.index.get_loc(fecha_entrada)
    if pos + 1 >= len(datos_h12):
        continue

    atr_actual = señales_con_atr.loc[fecha_entrada, "ATR_14"]
    if atr_actual <= umbral_bajo:
        grupo = tabla_sl_tp["Baja"]
    elif atr_actual <= umbral_alto:
        grupo = tabla_sl_tp["Media"]
    else:
        grupo = tabla_sl_tp["Alta"]

    nivel_sl = grupo["SL"]
    nivel_tp = grupo["TP"]
    precio_entrada = datos_h12.loc[fecha_entrada, "Close"]
    ventana = datos_h12.iloc[pos+1:]

    resultado = None
    retorno_op = None
    fecha_salida = None

    for fecha_v, vela in ventana.iterrows():
        if (vela["Low"] - precio_entrada) / precio_entrada <= nivel_sl:
            resultado, retorno_op, fecha_salida = "SL", nivel_sl, fecha_v
            break
        elif (vela["High"] - precio_entrada) / precio_entrada >= nivel_tp:
            resultado, retorno_op, fecha_salida = "TP", nivel_tp, fecha_v
            break

    if resultado is None:
        continue

    posiciones_abiertas_actual.append(fecha_salida)
    resultados_max3.append({"resultado": resultado, "retorno": retorno_op})

tabla_max3 = pd.DataFrame(resultados_max3)
años_cubiertos = (señales_ordenadas.index.max() - señales_ordenadas.index.min()).days / 365

print(f"Operaciones ejecutadas (máx 3 simultáneas): {len(tabla_max3)}")
print(tabla_max3["resultado"].value_counts())
print(f"Win rate: {(tabla_max3['retorno']>0).mean():.2%}")
print(f"Retorno promedio: {tabla_max3['retorno'].mean():.4%}")
print(f"Frecuencia: {len(tabla_max3)/años_cubiertos:.1f} operaciones/año ({len(tabla_max3)/años_cubiertos/52:.2f}/semana)")


Operaciones ejecutadas (máx 3 simultáneas): 160
resultado
TP    123
SL     37
Name: count, dtype: int64
Win rate: 76.88%
Retorno promedio: 0.9408%
Frecuencia: 81.3 operaciones/año (1.56/semana)


In [ ]:
capital = 1000
riesgo_por_operacion = 0.01
capital_historial = [capital]

posiciones_abiertas_actual = []
señales_ordenadas_v2 = señales_con_atr.sort_index()

for fecha_entrada in señales_ordenadas_v2.index:
    posiciones_abiertas_actual = [p for p in posiciones_abiertas_actual if p["fecha_salida"] > fecha_entrada]

    if len(posiciones_abiertas_actual) >= 3:
        continue

    pos = datos_h12.index.get_loc(fecha_entrada)
    if pos + 1 >= len(datos_h12):
        continue

    atr_actual = señales_con_atr.loc[fecha_entrada, "ATR_14"]
    if atr_actual <= umbral_bajo:
        grupo = tabla_sl_tp["Baja"]
    elif atr_actual <= umbral_alto:
        grupo = tabla_sl_tp["Media"]
    else:
        grupo = tabla_sl_tp["Alta"]

    nivel_sl, nivel_tp = grupo["SL"], grupo["TP"]
    precio_entrada = datos_h12.loc[fecha_entrada, "Close"]
    ventana = datos_h12.iloc[pos+1:]

    resultado, r_multiple, fecha_salida = None, None, None
    for fecha_v, vela in ventana.iterrows():
        if (vela["Low"]-precio_entrada)/precio_entrada <= nivel_sl:
            resultado, r_multiple, fecha_salida = "SL", -1.0, fecha_v
            break
        elif (vela["High"]-precio_entrada)/precio_entrada >= nivel_tp:
            resultado, r_multiple, fecha_salida = "TP", nivel_tp/abs(nivel_sl), fecha_v
            break

    if resultado is None:
        continue

    dinero_arriesgado = capital * riesgo_por_operacion
    resultado_dolares = dinero_arriesgado * r_multiple
    capital = capital + resultado_dolares

    posiciones_abiertas_actual.append({"fecha_salida": fecha_salida})
    capital_historial.append(capital)

print(f"Capital inicial: $1,000")
print(f"Capital final (máx 3 simultáneas, 1% cada una): ${capital:,.2f}")
print(f"Retorno total: {(capital/1000 - 1):.2%}")

capital_serie = pd.Series(capital_historial)
maximo_hist = capital_serie.cummax()
drawdown_serie = (capital_serie / maximo_hist) - 1
print(f"Drawdown máximo (esta versión, con 3 simultáneas resuelto): {drawdown_serie.min():.2%}")


Capital inicial: $1,000
Capital final (máx 3 simultáneas, 1% cada una): $1,705.26
Retorno total: 70.53%
Drawdown máximo (esta versión, con 3 simultáneas resuelto): -7.58%


In [ ]:
def simular_riesgo(riesgo_pct):
    capital = 1000
    capital_historial = [capital]
    posiciones_abiertas_actual = []

    for fecha_entrada in señales_ordenadas_v2.index:
        posiciones_abiertas_actual = [p for p in posiciones_abiertas_actual if p["fecha_salida"] > fecha_entrada]
        if len(posiciones_abiertas_actual) >= 3:
            continue

        pos = datos_h12.index.get_loc(fecha_entrada)
        if pos + 1 >= len(datos_h12):
            continue

        atr_actual = señales_con_atr.loc[fecha_entrada, "ATR_14"]
        if atr_actual <= umbral_bajo:
            grupo = tabla_sl_tp["Baja"]
        elif atr_actual <= umbral_alto:
            grupo = tabla_sl_tp["Media"]
        else:
            grupo = tabla_sl_tp["Alta"]

        nivel_sl, nivel_tp = grupo["SL"], grupo["TP"]
        precio_entrada = datos_h12.loc[fecha_entrada, "Close"]
        ventana = datos_h12.iloc[pos+1:]

        resultado, r_multiple, fecha_salida = None, None, None
        for fecha_v, vela in ventana.iterrows():
            if (vela["Low"]-precio_entrada)/precio_entrada <= nivel_sl:
                resultado, r_multiple, fecha_salida = "SL", -1.0, fecha_v
                break
            elif (vela["High"]-precio_entrada)/precio_entrada >= nivel_tp:
                resultado, r_multiple, fecha_salida = "TP", nivel_tp/abs(nivel_sl), fecha_v
                break

        if resultado is None:
            continue

        dinero_arriesgado = capital * riesgo_pct
        capital = capital + (dinero_arriesgado * r_multiple)
        posiciones_abiertas_actual.append({"fecha_salida": fecha_salida})
        capital_historial.append(capital)

    serie = pd.Series(capital_historial)
    dd = ((serie / serie.cummax()) - 1).min()
    return capital, dd

print(f"{'Riesgo':<10}{'Capital final':<18}{'Retorno':<12}{'Drawdown máx':<15}")
for riesgo in [0.01, 0.02, 0.03, 0.05, 0.10]:
    capital_f, drawdown_f = simular_riesgo(riesgo)
    print(f"{riesgo:<10.0%}${capital_f:<17,.2f}{(capital_f/1000-1):<12.2%}{drawdown_f:<15.2%}")

Riesgo    Capital final     Retorno     Drawdown máx   
1%        $1,705.26         70.53%      -7.58%         
2%        $2,876.49         187.65%     -14.69%        
3%        $4,799.92         379.99%     -21.34%        
5%        $12,939.95        1194.00%    -33.37%        
10%       $127,989.16       12698.92%   -56.93%        


In [ ]:
def simular_riesgo_con_historial(riesgo_pct):
    capital = 1000
    historial = [{"fecha": señales_ordenadas_v2.index.min(), "capital": capital, "resultado": "inicio"}]
    posiciones_abiertas_actual = []

    for fecha_entrada in señales_ordenadas_v2.index:
        posiciones_abiertas_actual = [p for p in posiciones_abiertas_actual if p["fecha_salida"] > fecha_entrada]
        if len(posiciones_abiertas_actual) >= 3:
            continue
        pos = datos_h12.index.get_loc(fecha_entrada)
        if pos + 1 >= len(datos_h12):
            continue
        atr_actual = señales_con_atr.loc[fecha_entrada, "ATR_14"]
        if atr_actual <= umbral_bajo:
            grupo = tabla_sl_tp["Baja"]
        elif atr_actual <= umbral_alto:
            grupo = tabla_sl_tp["Media"]
        else:
            grupo = tabla_sl_tp["Alta"]
        nivel_sl, nivel_tp = grupo["SL"], grupo["TP"]
        precio_entrada = datos_h12.loc[fecha_entrada, "Close"]
        ventana = datos_h12.iloc[pos+1:]
        resultado, r_multiple, fecha_salida = None, None, None
        for fecha_v, vela in ventana.iterrows():
            if (vela["Low"]-precio_entrada)/precio_entrada <= nivel_sl:
                resultado, r_multiple, fecha_salida = "SL", -1.0, fecha_v
                break
            elif (vela["High"]-precio_entrada)/precio_entrada >= nivel_tp:
                resultado, r_multiple, fecha_salida = "TP", nivel_tp/abs(nivel_sl), fecha_v
                break
        if resultado is None:
            continue
        dinero_arriesgado = capital * riesgo_pct
        capital = capital + (dinero_arriesgado * r_multiple)
        posiciones_abiertas_actual.append({"fecha_salida": fecha_salida})
        historial.append({"fecha": fecha_entrada, "capital": capital, "resultado": resultado})

    return pd.DataFrame(historial)

historial_10pct = simular_riesgo_con_historial(0.10)
historial_10pct["maximo"] = historial_10pct["capital"].cummax()
historial_10pct["drawdown"] = (historial_10pct["capital"] / historial_10pct["maximo"]) - 1

idx_peor = historial_10pct["drawdown"].idxmin()
idx_pico_previo = historial_10pct.loc[:idx_peor][historial_10pct["capital"] == historial_10pct.loc[:idx_peor, "maximo"].iloc[-1]].index[-1]

print("Secuencia real desde el pico hasta el peor momento:")
print(historial_10pct.loc[idx_pico_previo:idx_peor][["fecha", "capital", "resultado", "drawdown"]])

Secuencia real desde el pico hasta el peor momento:
                        fecha        capital resultado  drawdown
138 2026-01-27 16:00:00-05:00  194818.496310        TP  0.000000
139 2026-04-17 05:00:00-04:00  175336.646679        SL -0.100000
140 2026-04-24 05:00:00-04:00  157802.982011        SL -0.190000
141 2026-04-29 17:00:00-04:00  166357.236345        TP -0.146091
142 2026-04-30 05:00:00-04:00  175375.203508        TP -0.099802
143 2026-05-05 17:00:00-04:00  157837.683157        SL -0.189822
144 2026-05-06 17:00:00-04:00  142053.914841        SL -0.270840
145 2026-05-11 05:00:00-04:00  127848.523357        SL -0.343756
146 2026-05-18 05:00:00-04:00  115063.671021        SL -0.409380
147 2026-05-20 05:00:00-04:00  103557.303919        SL -0.468442
148 2026-05-26 17:00:00-04:00   93201.573527        SL -0.521598
149 2026-06-11 05:00:00-04:00   98253.885937        TP -0.495664
150 2026-06-11 17:00:00-04:00  103580.076347        TP -0.468325
151 2026-06-14 17:00:00-04:00   93222.

C:\Users\HP\AppData\Local\Temp\ipykernel_7184\3613214240.py:45: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  idx_pico_previo = historial_10pct.loc[:idx_peor][historial_10pct["capital"] == historial_10pct.loc[:idx_peor, "maximo"].iloc[-1]].index[-1]


In [ ]:
np.random.seed(42)
n_sim = 10000
drawdowns_bootstrap_corregido = []

retornos_r = tabla_max3["retorno"].values

for i in range(n_sim):
    muestra = np.random.choice(retornos_r, size=len(retornos_r), replace=True)
    capital_sim = 1000
    capital_curva = [capital_sim]
    for r in muestra:
        r_multiple_sim = 1.0 if r > 0 else -1.0
        capital_sim = capital_sim * (1 + 0.01 * r_multiple_sim)
        capital_curva.append(capital_sim)
    capital_curva = pd.Series(capital_curva)
    dd = ((capital_curva / capital_curva.cummax()) - 1).min()
    drawdowns_bootstrap_corregido.append(dd)

drawdowns_bootstrap_corregido = np.array(drawdowns_bootstrap_corregido)
print(f"Bootstrap (máx 3 simultáneas, 1% riesgo) — distribución de peores drawdowns posibles:")
print(f"  Promedio: {drawdowns_bootstrap_corregido.mean():.2%}")
print(f"  Percentil 95 (peor 5% de escenarios): {np.percentile(drawdowns_bootstrap_corregido, 5):.2%}")
print(f"  Percentil 99 (peor 1% de escenarios): {np.percentile(drawdowns_bootstrap_corregido, 1):.2%}")
print(f"\nDrawdown REAL observado (con 1% de riesgo): -7.58%")

Bootstrap (máx 3 simultáneas, 1% riesgo) — distribución de peores drawdowns posibles:
  Promedio: -3.39%
  Percentil 95 (peor 5% de escenarios): -4.94%
  Percentil 99 (peor 1% de escenarios): -6.79%

Drawdown REAL observado (con 1% de riesgo): -7.58%


In [ ]:
for riesgo in [0.01, 0.02, 0.03, 0.05, 0.10]:
    hist = simular_riesgo_con_historial(riesgo)
    hist["maximo"] = hist["capital"].cummax()
    hist["drawdown"] = (hist["capital"] / hist["maximo"]) - 1
    peor_dd = hist["drawdown"].min()
    print(f"Riesgo {riesgo:.0%}: peor drawdown = {peor_dd:.2%}")

Riesgo 1%: peor drawdown = -7.58%
Riesgo 2%: peor drawdown = -14.69%
Riesgo 3%: peor drawdown = -21.34%
Riesgo 5%: peor drawdown = -33.37%
Riesgo 10%: peor drawdown = -56.93%


In [ ]:
hist_1pct = simular_riesgo_con_historial(0.01)
hist_1pct["maximo"] = hist_1pct["capital"].cummax()
hist_1pct["bajo_el_agua"] = hist_1pct["capital"] < hist_1pct["maximo"]

pct_tiempo_bajo_agua = hist_1pct["bajo_el_agua"].mean()
print(f"% de operaciones donde estabas por debajo de tu máximo histórico: {pct_tiempo_bajo_agua:.1%}")

% de operaciones donde estabas por debajo de tu máximo histórico: 50.3%


In [ ]:
# A) La peor racha REAL de pérdidas consecutivas, en tus 160 operaciones reales
resultados_secuencia = tabla_max3["resultado"].values
racha_actual, peor_racha_real = 0, 0
for r in resultados_secuencia:
    racha_actual = racha_actual + 1 if r == "SL" else 0
    peor_racha_real = max(peor_racha_real, racha_actual)
print(f"A) Peor racha REAL de pérdidas seguidas (en tus 160 operaciones): {peor_racha_real}")

# B) Distribución de "peor racha posible" simulando 10,000 veces con tu win rate real
win_rate_max3 = (tabla_max3["retorno"] > 0).mean()
np.random.seed(42)
n_sim = 10000
peores_rachas_sim = np.zeros(n_sim, dtype=int)
for i in range(n_sim):
    sim = np.random.random(len(tabla_max3)) > win_rate_max3
    racha, peor = 0, 0
    for r in sim:
        racha = racha + 1 if r else 0
        peor = max(peor, racha)
    peores_rachas_sim[i] = peor

print(f"\nB) Distribución de peor racha, en 10,000 simulaciones:")
print(f"   Promedio: {peores_rachas_sim.mean():.1f}")
print(f"   Percentil 95: {int(np.percentile(peores_rachas_sim, 95))}")
print(f"   Percentil 99: {int(np.percentile(peores_rachas_sim, 99))}")
print(f"   MÁXIMO absoluto visto en las 10,000 simulaciones: {peores_rachas_sim.max()}")

# C) El PEOR drawdown absoluto entre las 10,000 simulaciones del bootstrap (no solo percentil 99)
print(f"\nC) Bootstrap de drawdown — el peor de los 10,000 escenarios simulados:")
print(f"   Peor caso absoluto (mínimo de toda la distribución): {drawdowns_bootstrap_corregido.min():.2%}")

A) Peor racha REAL de pérdidas seguidas (en tus 160 operaciones): 6

B) Distribución de peor racha, en 10,000 simulaciones:
   Promedio: 3.2
   Percentil 95: 5
   Percentil 99: 6
   MÁXIMO absoluto visto en las 10,000 simulaciones: 9

C) Bootstrap de drawdown — el peor de los 10,000 escenarios simulados:
   Peor caso absoluto (mínimo de toda la distribución): -10.50%


In [ ]:
resultados_max3 = tabla_max3["resultado"].values
racha_actual, mejor_inicio, mejor_fin, peor_racha = 0, 0, 0, 0
inicio_temp = 0

for i, r in enumerate(resultados_max3):
    if r == "SL":
        if racha_actual == 0:
            inicio_temp = i
        racha_actual += 1
        if racha_actual > peor_racha:
            peor_racha = racha_actual
            mejor_inicio, mejor_fin = inicio_temp, i
    else:
        racha_actual = 0

print(f"Peor racha real: operaciones #{mejor_inicio} a #{mejor_fin} (6 pérdidas seguidas)")
print(f"Drawdown del historial en esos mismos índices:")
print(hist_1pct.iloc[mejor_inicio:mejor_fin+2][["fecha", "capital", "resultado", "drawdown"]])

Peor racha real: operaciones #142 a #147 (6 pérdidas seguidas)
Drawdown del historial en esos mismos índices:
                        fecha      capital resultado  drawdown
142 2026-04-30 05:00:00-04:00  1750.773425        TP -0.009245
143 2026-05-05 17:00:00-04:00  1733.265691        SL -0.019153
144 2026-05-06 17:00:00-04:00  1715.933034        SL -0.028961
145 2026-05-11 05:00:00-04:00  1698.773704        SL -0.038672
146 2026-05-18 05:00:00-04:00  1681.785967        SL -0.048285
147 2026-05-20 05:00:00-04:00  1664.968107        SL -0.057802
148 2026-05-26 17:00:00-04:00  1648.318426        SL -0.067224


In [ ]:
hist_1pct["drawdown"] = (hist_1pct["capital"] / hist_1pct["maximo"]) - 1

print(f"Peor racha real: operaciones #{mejor_inicio} a #{mejor_fin} (6 pérdidas seguidas)")
print(f"Drawdown del historial en esos mismos índices:")
print(hist_1pct.iloc[mejor_inicio:mejor_fin+2][["fecha", "capital", "resultado", "drawdown"]])

Peor racha real: operaciones #142 a #147 (6 pérdidas seguidas)
Drawdown del historial en esos mismos índices:
                        fecha      capital resultado  drawdown
142 2026-04-30 05:00:00-04:00  1750.773425        TP -0.009245
143 2026-05-05 17:00:00-04:00  1733.265691        SL -0.019153
144 2026-05-06 17:00:00-04:00  1715.933034        SL -0.028961
145 2026-05-11 05:00:00-04:00  1698.773704        SL -0.038672
146 2026-05-18 05:00:00-04:00  1681.785967        SL -0.048285
147 2026-05-20 05:00:00-04:00  1664.968107        SL -0.057802
148 2026-05-26 17:00:00-04:00  1648.318426        SL -0.067224


In [ ]:
print(hist_1pct.iloc[147:156][["fecha", "capital", "resultado", "drawdown"]])

                        fecha      capital resultado  drawdown
147 2026-05-20 05:00:00-04:00  1664.968107        SL -0.057802
148 2026-05-26 17:00:00-04:00  1648.318426        SL -0.067224
149 2026-06-11 05:00:00-04:00  1657.253704        TP -0.062168
150 2026-06-11 17:00:00-04:00  1666.237419        TP -0.057084
151 2026-06-14 17:00:00-04:00  1649.575044        SL -0.066513
152 2026-06-15 17:00:00-04:00  1633.079294        SL -0.075848
153 2026-06-25 05:00:00-04:00  1641.931963        TP -0.070838
154 2026-06-25 17:00:00-04:00  1650.832621        TP -0.065801
155 2026-07-01 05:00:00-04:00  1659.781528        TP -0.060737


In [ ]:
np.random.seed(42)
n_sim = 10000
resultados_labels = tabla_max3["resultado"].values

rachas_sim = []
drawdowns_sim = []

for i in range(n_sim):
    muestra = np.random.choice(resultados_labels, size=len(resultados_labels), replace=True)
    capital_sim = [1000]
    racha, peor_racha_sim = 0, 0
    for r in muestra:
        if r == "SL":
            racha += 1
            capital_sim.append(capital_sim[-1] * (1 - 0.01))
        else:
            peor_racha_sim = max(peor_racha_sim, racha)
            racha = 0
            capital_sim.append(capital_sim[-1] * (1 + 0.01))
    peor_racha_sim = max(peor_racha_sim, racha)

    serie_sim = pd.Series(capital_sim)
    dd_sim = ((serie_sim / serie_sim.cummax()) - 1).min()

    rachas_sim.append(peor_racha_sim)
    drawdowns_sim.append(dd_sim)

rachas_sim = np.array(rachas_sim)
drawdowns_sim = np.array(drawdowns_sim)

correlacion = np.corrcoef(rachas_sim, drawdowns_sim)[0, 1]
print(f"Correlación entre peor racha y peor drawdown: {correlacion:.3f}")

peor_10pct_mask = drawdowns_sim <= np.percentile(drawdowns_sim, 10)
print(f"Racha promedio EN el peor 10% de drawdowns: {rachas_sim[peor_10pct_mask].mean():.2f}")
print(f"Racha promedio en TODAS las simulaciones: {rachas_sim.mean():.2f}")

Correlación entre peor racha y peor drawdown: -0.860
Racha promedio EN el peor 10% de drawdowns: 4.52
Racha promedio en TODAS las simulaciones: 3.18


In [ ]:
np.random.seed(42)
n_sim = 10000
resultados_labels = tabla_max3["resultado"].values

drawdown_de_la_peor_racha = []

for i in range(n_sim):
    muestra = np.random.choice(resultados_labels, size=len(resultados_labels), replace=True)

    racha_actual = 0
    peor_racha_sim = 0
    racha_mas_larga_actual = 0

    for r in muestra:
        if r == "SL":
            racha_actual += 1
            if racha_actual > peor_racha_sim:
                peor_racha_sim = racha_actual
        else:
            racha_actual = 0

    drawdown_racha = 1 - (1 - 0.01) ** peor_racha_sim
    drawdown_de_la_peor_racha.append(-drawdown_racha)

drawdown_de_la_peor_racha = np.array(drawdown_de_la_peor_racha)

print("Drawdown generado ESPECÍFICAMENTE por la peor racha de cada simulación:")
print(f"  Promedio: {drawdown_de_la_peor_racha.mean():.2%}")
print(f"  Percentil 95: {np.percentile(drawdown_de_la_peor_racha, 5):.2%}")
print(f"  Percentil 99: {np.percentile(drawdown_de_la_peor_racha, 1):.2%}")
print(f"  Máximo absoluto (peor de las 10,000 simulaciones): {drawdown_de_la_peor_racha.min():.2%}")

print(f"\nComparación con tu caso real:")
print(f"  Tu peor racha real fue de 6 pérdidas -> drawdown de esa racha específica: {1-(1-0.01)**6:.2%} (negativo)")


Drawdown generado ESPECÍFICAMENTE por la peor racha de cada simulación:
  Promedio: -3.14%
  Percentil 95: -4.90%
  Percentil 99: -5.85%
  Máximo absoluto (peor de las 10,000 simulaciones): -7.73%

Comparación con tu caso real:
  Tu peor racha real fue de 6 pérdidas -> drawdown de esa racha específica: 5.85% (negativo)


In [ ]:
def simular_riesgo_por_pico(riesgo_pct):
    capital = 1000
    pico_maximo = 1000
    capital_historial = [capital]
    posiciones_abiertas_actual = []

    for fecha_entrada in señales_ordenadas_v2.index:
        posiciones_abiertas_actual = [p for p in posiciones_abiertas_actual if p["fecha_salida"] > fecha_entrada]
        if len(posiciones_abiertas_actual) >= 3:
            continue

        pos = datos_h12.index.get_loc(fecha_entrada)
        if pos + 1 >= len(datos_h12):
            continue

        atr_actual = señales_con_atr.loc[fecha_entrada, "ATR_14"]
        if atr_actual <= umbral_bajo:
            grupo = tabla_sl_tp["Baja"]
        elif atr_actual <= umbral_alto:
            grupo = tabla_sl_tp["Media"]
        else:
            grupo = tabla_sl_tp["Alta"]

        nivel_sl, nivel_tp = grupo["SL"], grupo["TP"]
        precio_entrada = datos_h12.loc[fecha_entrada, "Close"]
        ventana = datos_h12.iloc[pos+1:]

        resultado, r_multiple, fecha_salida = None, None, None
        for fecha_v, vela in ventana.iterrows():
            if (vela["Low"]-precio_entrada)/precio_entrada <= nivel_sl:
                resultado, r_multiple, fecha_salida = "SL", -1.0, fecha_v
                break
            elif (vela["High"]-precio_entrada)/precio_entrada >= nivel_tp:
                resultado, r_multiple, fecha_salida = "TP", nivel_tp/abs(nivel_sl), fecha_v
                break

        if resultado is None:
            continue

        dinero_arriesgado = pico_maximo * riesgo_pct
        capital = capital + (dinero_arriesgado * r_multiple)
        pico_maximo = max(pico_maximo, capital)

        posiciones_abiertas_actual.append({"fecha_salida": fecha_salida})
        capital_historial.append(capital)

    serie = pd.Series(capital_historial)
    dd = ((serie / serie.cummax()) - 1).min()
    return capital, dd

capital_pico, dd_pico = simular_riesgo_por_pico(0.01)
print(f"VARIANTE 'último pico' (1% de riesgo):")
print(f"  Capital final: ${capital_pico:,.2f}")
print(f"  Drawdown máximo: {dd_pico:.2%}")
print(f"\nVARIANTE 'capital actual' (la que ya usamos, 1% de riesgo):")
print(f"  Capital final: $1,705.26")
print(f"  Drawdown máximo: -7.58%")

VARIANTE 'último pico' (1% de riesgo):
  Capital final: $1,709.53
  Drawdown máximo: -7.83%

VARIANTE 'capital actual' (la que ya usamos, 1% de riesgo):
  Capital final: $1,705.26
  Drawdown máximo: -7.58%


In [ ]:
r_multiples_reales = np.where(tabla_max3["resultado"]=="TP",
                                tabla_sl_tp["Media"]["TP"]/abs(tabla_sl_tp["Media"]["SL"]),
                                -1.0)

win_rate_general = (tabla_max3["resultado"]=="TP").mean()
r_ratio_promedio_ponderado = tabla_max3[tabla_max3["resultado"]=="TP"]["retorno"].mean() / abs(tabla_max3[tabla_max3["resultado"]=="SL"]["retorno"].mean())

expectativa_general = (win_rate_general * r_ratio_promedio_ponderado) - ((1-win_rate_general) * 1)

print(f"Win rate general: {win_rate_general:.2%}")
print(f"R:R promedio ponderado (mezclando los 3 grupos según su frecuencia real): 1 : {r_ratio_promedio_ponderado:.2f}")
print(f"Expectativa por operación: {expectativa_general:.3f}R")
print(f"\nEsto significa: por cada $1 que arriesgas, en promedio ganas ${expectativa_general:.2f} netos")

Win rate general: 76.88%
R:R promedio ponderado (mezclando los 3 grupos según su frecuencia real): 1 : 0.67
Expectativa por operación: 0.283R

Esto significa: por cada $1 que arriesgas, en promedio ganas $0.28 netos
